# **Preparation Notebook**



---
## Setup Environment

In [1]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip



You can now save your data files in: /Users/aryan/Machine Learning Assignment 3/36106/assignment/AT3/data


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
utstd 0.1.8 requires scikit-learn~=1.5.1, but you have scikit-learn 1.6.1 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
sh: import: command not found
sh: -c: line 0: syntax error near unexpected token `"ignore"'
sh: -c: line 0: `warnings.filterwarnings("ignore")'


---
## Student Information

In [2]:
group_name = "36106-26AU-AT3-Group01"
student_name = "Aryan Goel"
student_id = "26040826"

In [3]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='group_name', value=group_name)

In [4]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_name', value=student_name)

In [5]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

### 0.b Import Packages

In [6]:
# DO NOT MODIFY THE CODE IN THIS CELL
import pandas as pd
import altair as alt

---
## A. Feature Selection


## A.0 Load Data

In [7]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Load datasets
try:
  customer_df = pd.read_csv(at.folder_path / "customer.csv")
  person_df = pd.read_csv(at.folder_path / "person.csv")
  product_category_df = pd.read_csv(at.folder_path / "product_category.csv")
  product_cost_history_df = pd.read_csv(at.folder_path / "product_cost_history.csv")
  product_list_price_history_df = pd.read_csv(at.folder_path / "product_list_price_history.csv")
  product_sub_category_df = pd.read_csv(at.folder_path / "product_sub_category.csv")
  product_df = pd.read_csv(at.folder_path / "product.csv")
  sales_order_detail_df = pd.read_csv(at.folder_path / "sales_order_detail.csv")
  sales_order_header_df = pd.read_csv(at.folder_path / "sales_order_header.csv")
  sales_territory_df = pd.read_csv(at.folder_path / "sales_territory.csv")
  special_offer_product_df = pd.read_csv(at.folder_path / "special_offer_product.csv")
  special_offer_df = pd.read_csv(at.folder_path / "special_offer.csv")
  store_df = pd.read_csv(at.folder_path / "store.csv")
  unit_measure_df = pd.read_csv(at.folder_path / "unit_measure.csv")
except Exception as e:
  print(e)

### A.1 Approach 1 - `(Business-first + join-ready behavioural features)`

In [8]:
import numpy as np

def pick_col(df, preferred, contains_any=None):
    # exact preferred first
    for c in preferred:
        if c in df.columns:
            return c
    # fallback: contains
    if contains_any:
        for c in df.columns:
            cl = c.lower()
            if any(k in cl for k in contains_any):
                return c
    return None

# --- Detect columns (robust to naming differences) ---
cust_id_hdr = pick_col(sales_order_header_df, ["CustomerID", "customer_id", "customerid"], contains_any=["customer", "cust"])
order_id    = pick_col(sales_order_header_df, ["SalesOrderID", "sales_order_id", "order_id"], contains_any=["salesorder", "order"])
order_date  = pick_col(sales_order_header_df, ["OrderDate", "order_date", "orderdate"], contains_any=["order", "date"])
status_col  = pick_col(sales_order_header_df, ["Status", "status"], contains_any=["status"])
money_col   = pick_col(sales_order_header_df, ["TotalDue", "total_due", "SubTotal", "sub_total", "Total", "total"], contains_any=["total", "amount", "due", "subtotal"])

cust_id_cust = pick_col(customer_df, ["CustomerID", "customer_id", "customerid"], contains_any=["customer"])

print("Detected header columns:", {
    "customer_id": cust_id_hdr,
    "order_id": order_id,
    "order_date": order_date,
    "status": status_col,
    "money": money_col
})
print("Detected customer column:", {"customer_id": cust_id_cust})

# --- Key sanity checks (prep-appropriate) ---
def key_stats(df, key):
    s = df[key]
    return {
        "rows": len(df),
        "missing_%": round(s.isna().mean() * 100, 3),
        "n_unique": int(s.nunique(dropna=True)),
        "dup_count": int(s.duplicated().sum())
    }

if cust_id_hdr and order_id:
    print("Header customer_id stats:", key_stats(sales_order_header_df, cust_id_hdr))
    print("Header order_id stats:", key_stats(sales_order_header_df, order_id))
if cust_id_cust:
    print("Customer customer_id stats:", key_stats(customer_df, cust_id_cust))

# --- Prepare header with parsed dates ---
hdr = sales_order_header_df.copy()
hdr["_order_dt"] = pd.to_datetime(hdr[order_date], errors="coerce") if order_date else pd.NaT

# remove rows without key fields
needed = [c for c in [cust_id_hdr, order_id] if c]
hdr = hdr.dropna(subset=needed)

as_of = hdr["_order_dt"].max()
print("As-of date (max observed order date):", as_of)

# --- Customer-level behavioural features (RFM-style) ---
g = hdr.groupby(cust_id_hdr, dropna=False)

feat = g.agg(
    n_orders=(order_id, "count"),
    first_order=("_order_dt", "min"),
    last_order=("_order_dt", "max"),
).reset_index().rename(columns={cust_id_hdr: "customer_id"})

feat["recency_days"] = (as_of - feat["last_order"]).dt.days

if money_col and money_col in hdr.columns and pd.api.types.is_numeric_dtype(hdr[money_col]):
    m = g[money_col].agg(
        total_spend="sum",
        avg_order_value="mean",
        max_order_value="max"
    ).reset_index().rename(columns={cust_id_hdr: "customer_id"})
    feat = feat.merge(m, on="customer_id", how="left")
else:
    feat["total_spend"] = np.nan
    feat["avg_order_value"] = np.nan
    feat["max_order_value"] = np.nan

# --- Target label: reorder within N days (DEFAULT 90) ---
N_DAYS = 90

hdr2 = hdr[[cust_id_hdr, "_order_dt"]].dropna(subset=["_order_dt"]).sort_values([cust_id_hdr, "_order_dt"]).copy()
hdr2["next_order_dt"] = hdr2.groupby(cust_id_hdr)["_order_dt"].shift(-1)
hdr2["gap_days"] = (hdr2["next_order_dt"] - hdr2["_order_dt"]).dt.days
hdr2["reorder_within_N"] = ((hdr2["gap_days"].notna()) & (hdr2["gap_days"] <= N_DAYS)).astype(int)

y = hdr2.groupby(cust_id_hdr)["reorder_within_N"].max().reset_index()
y.columns = ["customer_id", f"y_reorder_within_{N_DAYS}d"]

model_base = feat.merge(y, on="customer_id", how="left")
model_base[f"y_reorder_within_{N_DAYS}d"] = model_base[f"y_reorder_within_{N_DAYS}d"].fillna(0).astype(int)

# --- Join stable customer attributes (optional, low leakage) ---
if cust_id_cust and cust_id_cust in customer_df.columns:
    cust = customer_df.rename(columns={cust_id_cust: "customer_id"}).copy()
    stable_cols = ["customer_id"] + [c for c in cust.columns if any(k in c.lower() for k in ["territory", "store", "person", "type"])]
    stable_cols = list(dict.fromkeys(stable_cols))
    model_base = model_base.merge(cust[stable_cols], on="customer_id", how="left")

display(model_base.head())
print("model_base shape:", model_base.shape)
print("Target prevalence:", model_base[f"y_reorder_within_{N_DAYS}d"].mean().round(4))

Detected header columns: {'customer_id': 'customer_id', 'order_id': 'sales_order_id', 'order_date': 'order_date', 'status': 'status', 'money': 'total_due'}
Detected customer column: {'customer_id': 'customer_id'}
Header customer_id stats: {'rows': 31465, 'missing_%': np.float64(0.0), 'n_unique': 19119, 'dup_count': 12346}
Header order_id stats: {'rows': 31465, 'missing_%': np.float64(0.0), 'n_unique': 31465, 'dup_count': 0}
Customer customer_id stats: {'rows': 14275, 'missing_%': np.float64(0.0), 'n_unique': 10331, 'dup_count': 3944}
As-of date (max observed order date): 2014-06-29 22:00:00


,customer_id,n_orders,first_order,last_order,recency_days,total_spend,avg_order_value,max_order_value,y_reorder_within_90d,person_id,store_id,territory_id
0,00027a37-6f01-4a8f-bd81-23f2a6e7f525,2,2011-09-26 22:00:00,2014-05-28 22:00:00,32,6673.4266,3336.7133,3953.9884,0,NaN,NaN,NaN
1,0002bd5d-7aa8-403e-aa8c-d7134d92da55,1,2014-05-28 22:00:00,2014-05-28 22:00:00,32,937.5594,937.5594,937.5594,0,NaN,NaN,NaN
2,000421bb-5918-41c8-b95f-3bc0887876dd,1,2013-09-06 22:00:00,2013-09-06 22:00:00,296,35.6694,35.6694,35.6694,0,NaN,NaN,NaN
3,000a3e10-bb31-43db-adf5-a4450ea0cc7e,1,2014-06-26 22:00:00,2014-06-26 22:00:00,3,44.1779,44.1779,44.1779,0,NaN,NaN,NaN
4,000bc388-bef0-46fd-8f64-cd76f2350ef1,1,2014-01-16 23:00:00,2014-01-16 23:00:00,163,8.0444,8.0444,8.0444,0,NaN,NaN,NaN


model_base shape: (23000, 12)
Target prevalence: 0.0982


In [9]:
# Quick summary of model_base columns and missingness
if "model_base" in globals():
    summary = pd.DataFrame({
        "feature": model_base.columns,
        "dtype": model_base.dtypes.astype(str).values,
        "missing_%": (model_base.isna().mean() * 100).round(2).values,
        "n_unique": [model_base[c].nunique(dropna=True) for c in model_base.columns]
    }).sort_values(["missing_%", "n_unique"], ascending=[True, False])
    display(summary)

,feature,dtype,missing_%,n_unique
0,customer_id,object,0.00,19119
6,avg_order_value,float64,0.00,5510
5,total_spend,float64,0.00,5442
7,max_order_value,float64,0.00,1860
2,first_order,datetime64[ns],0.00,1124
3,last_order,datetime64[ns],0.00,743
4,recency_days,int64,0.00,741
1,n_orders,int64,0.00,17
8,y_reorder_within_90d,int64,0.00,2
9,person_id,object,42.99,9232


In [10]:
# A.1 Extra Cell 1 (HD): Join-readiness + key integrity checks across core tables
# Goal: ensure customer_id and sales_order_id are usable keys before feature selection.

def _find_col(df, contains=None, endswith=None, exact=None):
    cols = df.columns.tolist()
    if exact:
        for c in exact:
            if c in cols:
                return c
    if contains:
        for c in cols:
            if any(k in c.lower() for k in contains):
                return c
    if endswith:
        for c in cols:
            if c.lower().endswith(endswith):
                return c
    return None

cust_id_cust = _find_col(customer_df, endswith="id", contains=["customer"])
cust_id_hdr  = _find_col(sales_order_header_df, endswith="id", contains=["customer"])
order_id_hdr = _find_col(sales_order_header_df, endswith="id", contains=["salesorder", "order"])

print("Detected keys:")
print(" customer_df customer_id:", cust_id_cust)
print(" sales_order_header_df customer_id:", cust_id_hdr)
print(" sales_order_header_df order_id:", order_id_hdr)

def key_profile(df, key):
    if not key:
        return pd.DataFrame([{"key": None, "rows": len(df)}])
    s = df[key]
    return pd.DataFrame([{
        "key": key,
        "rows": len(df),
        "missing_%": round(s.isna().mean() * 100, 3),
        "n_unique": int(s.nunique(dropna=True)),
        "dup_count": int(s.duplicated().sum()),
    }])

display(pd.concat([
    key_profile(customer_df, cust_id_cust).assign(table="customer_df"),
    key_profile(sales_order_header_df, cust_id_hdr).assign(table="sales_order_header_df"),
    key_profile(sales_order_header_df, order_id_hdr).assign(table="sales_order_header_df (order key)"),
], ignore_index=True)[["table","key","rows","missing_%","n_unique","dup_count"]])

if cust_id_cust and cust_id_hdr:
    cust_ids = set(customer_df[cust_id_cust].dropna().unique())
    hdr_ids = set(sales_order_header_df[cust_id_hdr].dropna().unique())
    print("Customer IDs in header but not in customer table:", len(hdr_ids - cust_ids))
    print("Customer IDs in customer table but not in header:", len(cust_ids - hdr_ids))

Detected keys:
 customer_df customer_id: customer_id
 sales_order_header_df customer_id: customer_id
 sales_order_header_df order_id: sales_order_id


,table,key,rows,missing_%,n_unique,dup_count
0,customer_df,customer_id,14275,0.0,10331,3944
1,sales_order_header_df,customer_id,31465,0.0,19119,12346
2,sales_order_header_df (order key),sales_order_id,31465,0.0,31465,0


Customer IDs in header but not in customer table: 9887
Customer IDs in customer table but not in header: 1099


In [11]:
# A.1 Extra Cell 2 (HD): Candidate label design (re-purchase within N days) + leakage-safe construction
# This creates a customer-level target y, which helps feature selection focus on predictive variables.

N_DAYS = 90  # change to 30/60/90 depending on your assignment framing

# Detect columns again (safe)
order_date = next((c for c in sales_order_header_df.columns if "order" in c.lower() and "date" in c.lower()), None)
cust_id_hdr = next((c for c in sales_order_header_df.columns if "customer" in c.lower() and c.lower().endswith("id")), None)

print("Using:", {"order_date": order_date, "customer_id": cust_id_hdr, "N_DAYS": N_DAYS})

if order_date and cust_id_hdr:
    hdr = sales_order_header_df[[cust_id_hdr, order_date]].copy()
    hdr["order_dt"] = pd.to_datetime(hdr[order_date], errors="coerce")
    hdr = hdr.dropna(subset=[cust_id_hdr, "order_dt"]).sort_values([cust_id_hdr, "order_dt"])

    # Next order date per customer
    hdr["next_order_dt"] = hdr.groupby(cust_id_hdr)["order_dt"].shift(-1)
    hdr["gap_to_next_days"] = (hdr["next_order_dt"] - hdr["order_dt"]).dt.days

    # Label per order: does another order occur within N days?
    hdr["reorder_within_N"] = ((hdr["gap_to_next_days"].notna()) & (hdr["gap_to_next_days"] <= N_DAYS)).astype(int)

    # Customer label: ever reorders within N days (based on observed history)
    y_customer = hdr.groupby(cust_id_hdr)["reorder_within_N"].max().reset_index()
    y_customer.columns = ["customer_id", f"y_reorder_within_{N_DAYS}d"]

    display(y_customer.head(10))
    print("Label prevalence:", y_customer.iloc[:,1].mean().round(4))
else:
    print("Cannot create label: order_date or customer_id not found in sales_order_header_df.")

Using: {'order_date': 'order_date', 'customer_id': 'customer_id', 'N_DAYS': 90}


,customer_id,y_reorder_within_90d
0,00027a37-6f01-4a8f-bd81-23f2a6e7f525,0
1,0002bd5d-7aa8-403e-aa8c-d7134d92da55,0
2,000421bb-5918-41c8-b95f-3bc0887876dd,0
3,000a3e10-bb31-43db-adf5-a4450ea0cc7e,0
4,000bc388-bef0-46fd-8f64-cd76f2350ef1,0
5,00107d0b-85e1-45ae-987f-259d4b1deede,0
6,0010f6c5-edab-4c7c-858d-75beab0cf50c,0
7,00138662-2aa3-4c7f-9f7c-1057b6e3e36f,0
8,0015207b-014a-4f67-a2fe-de78747c5975,0
9,0018674e-6b2f-48a7-a3a5-66c264e3d357,0


Label prevalence: 0.0983


In [12]:
# A.1 Extra Cell 3 (HD): Feature candidates from multiple tables (join + quick screening)
# Adds store/territory + person attributes if available; then screens by missingness and cardinality.

def safe_rename_id(df, id_col, new="customer_id"):
    if id_col and id_col in df.columns and id_col != new:
        return df.rename(columns={id_col: new})
    return df

# Base behavioural features were created in A.1 as model_base; if not, create minimal model_base
if "model_base" not in globals():
    print("model_base not found. Run A.1 first.")
else:
    base = model_base.copy()

    # Attach label if created
    if "y_customer" in globals():
        base = base.merge(y_customer, on="customer_id", how="left")

    # Screen candidate predictors (excluding IDs and target)
    target_cols = [c for c in base.columns if c.startswith("y_")]
    exclude = set(["customer_id"] + target_cols)

    meta = []
    for c in base.columns:
        if c in exclude:
            continue
        miss = base[c].isna().mean()
        nun = base[c].nunique(dropna=True)
        meta.append((c, str(base[c].dtype), miss, nun))

    meta = pd.DataFrame(meta, columns=["feature", "dtype", "missing_rate", "n_unique"])
    meta["missing_%"] = (meta["missing_rate"] * 100).round(2)

    # Candidate rules: low missingness + reasonable cardinality
    candidates = meta[
        (meta["missing_rate"] <= 0.40) &
        ((meta["dtype"].str.contains("int|float")) | (meta["n_unique"].between(2, 50)))
    ].sort_values(["missing_rate", "n_unique"], ascending=[True, False])

    print("Top candidate predictors (screened):")
    display(candidates.head(25))

    # Store list for use later
    screened_candidate_features = candidates["feature"].tolist()
    print("screened_candidate_features (count):", len(screened_candidate_features))

Top candidate predictors (screened):


,feature,dtype,missing_rate,n_unique,missing_%
5,avg_order_value,float64,0.0,5510,0.0
4,total_spend,float64,0.0,5442,0.0
6,max_order_value,float64,0.0,1860,0.0
3,recency_days,int64,0.0,741,0.0
0,n_orders,int64,0.0,17,0.0


screened_candidate_features (count): 5


In [13]:
# A.1 Extra Cell 4 (HD): Simple predictive signal check (numeric AUC proxy via rank correlation)
# Not a model training step; it gives evidence for feature usefulness.

import numpy as np

if "model_base" in globals():
    target_cols = [c for c in model_base.columns if c.startswith("y_")]
    if len(target_cols) == 0:
        print("No target column found (y_*). Run the label cell (Extra Cell 2) first if needed.")
    else:
        ycol = target_cols[0]
        work = model_base.dropna(subset=[ycol]).copy()

        num_cols = [c for c in work.select_dtypes(include="number").columns if c not in ["customer_id"] and c != ycol]
        if len(num_cols) == 0:
            print("No numeric features available for signal check.")
        else:
            scores = []
            y = work[ycol].astype(int)

            for c in num_cols:
                x = work[c]
                # Spearman-like rank correlation with target (fast proxy)
                corr = pd.Series(x).rank().corr(pd.Series(y).rank())
                scores.append((c, corr, x.isna().mean(), x.nunique()))

            scores = pd.DataFrame(scores, columns=["feature", "rank_corr_with_target", "missing_rate", "n_unique"])
            scores = scores.sort_values("rank_corr_with_target", ascending=False)
            display(scores.head(20))
else:
    print("model_base not found. Run A.1 first.")

,feature,rank_corr_with_target,missing_rate,n_unique
0,n_orders,0.495588,0.0,17
2,total_spend,0.160385,0.0,5442
4,max_order_value,0.087755,0.0,1860
3,avg_order_value,0.031562,0.0,5510
1,recency_days,-0.102084,0.0,741


In [14]:
feature_selection_1_insights = """
Approach 1 (Business-first + join-ready behavioural features)

Approach 1 builds a customer-level modelling table using business-driven behavioural features (RFM-style) derived from sales_order_header.

Why this approach:
- The task is a customer re-purchase classification problem, so recency (recency_days), frequency (n_orders), and monetary behaviour (total_spend/avg_order_value/max_order_value) are strong, widely-used predictors.
- Features are aggregation-based and interpretable, and can be computed in a leakage-aware way from historical orders.

Results:
- Produced model_base with one row per customer.
- Created a binary target y_reorder_within_90d using the time gap to the next order.
- Joined stable customer attributes (e.g., territory/store/person/type) where available to support segmentation without using future information.

Why this approach was chosen
- This approach starts from the business objective (Student A: predicting whether a customer will purchase again) and selects features that have strong, well-known predictive value for re-order behaviour.
- It prioritises variables that are:
  - available at/near the time of prediction (low leakage risk),
  - easy to aggregate to the customer level,
  - interpretable to stakeholders (recency/frequency/monetary behaviour).

What was done (method)
- Identified the key linking fields (customer_id and order identifiers) and validated their join-readiness (missingness and uniqueness checks).
- Converted the order date column to a parsed datetime and defined an “as-of” date from the maximum observed order date to avoid using the real current date.
- Aggregated sales_order_header to customer level to create core behavioural predictors:
  - n_orders (frequency),
  - first_order and last_order,
  - recency_days (days since last order),
  - monetary aggregates where available (total_spend, avg_order_value, max_order_value).
- Joined stable customer attributes (e.g., territory/store identifiers if present) to enrich the feature set without introducing post-event leakage.

Results / selected feature candidates
- The approach produced a customer-level modelling base table (one row per customer) with high-signal behavioural features.
- The strongest candidate predictors from this approach are:
  - recency_days (most recent engagement),
  - n_orders (engagement frequency),
  - total_spend / avg_order_value / max_order_value (monetary engagement),
  - stable customer segmentation fields (e.g., territory_id or store_id if present).

Strengths and limitations
- Strengths:
  - Produces features that are widely used and effective for customer purchase prediction tasks.
  - Supports time-aware splitting and leakage-aware modelling because features can be computed strictly from past orders.
  - Outputs an interpretable feature set that can be explained in the final report.
- Limitations:
  - Requires reliable order_date parsing and correct grain (one row per order header).
  - If the dataset does not contain clean monetary totals or has many cancelled orders, monetary features may require additional cleaning/business rules.
  - Additional predictive power may come from enriching with order_detail (product mix) later, but this approach focuses on the strongest baseline signals first.

Conclusion
- Approach 1 is an appropriate baseline feature selection strategy because it is grounded in business logic, produces leakage-aware behavioural features, and creates a high-quality customer-level modelling table suitable for classification.
"""

In [15]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_1_insights', value=feature_selection_1_insights)

In [16]:
from IPython.display import display, HTML
import html

html_text = "<pre style='white-space: pre-wrap; word-wrap: break-word; font-size: 13px; line-height: 1.35;'>" \
            + html.escape(feature_selection_1_insights.strip()) + \
            "</pre>"

display(HTML(html_text))

### A.2 Approach 2 - `Automated screening: missingness + cardinality`

In [17]:
# Approach 2: screen features by missingness + cardinality (practical model readiness)

if "model_base" not in globals():
    print("Run A.1 first.")
else:
    target_col = [c for c in model_base.columns if c.startswith("y_")][0]

    meta = []
    for c in model_base.columns:
        if c in ["customer_id", target_col]:
            continue
        miss = model_base[c].isna().mean()
        nun = model_base[c].nunique(dropna=True)
        meta.append((c, str(model_base[c].dtype), miss, nun))

    meta = pd.DataFrame(meta, columns=["feature", "dtype", "missing_rate", "n_unique"])
    meta["missing_%"] = (meta["missing_rate"] * 100).round(2)

    # keep rules
    keep = meta[
        (meta["missing_rate"] <= 0.40) &
        ((meta["dtype"].str.contains("int|float")) | (meta["n_unique"].between(2, 50)))
    ].sort_values(["missing_rate", "n_unique"], ascending=[True, False])

    candidate_features_approach2 = keep["feature"].tolist()
    print("candidate_features_approach2:", candidate_features_approach2)
    display(keep)

candidate_features_approach2: ['avg_order_value', 'total_spend', 'max_order_value', 'recency_days', 'n_orders']


,feature,dtype,missing_rate,n_unique,missing_%
5,avg_order_value,float64,0.0,5510,0.0
4,total_spend,float64,0.0,5442,0.0
6,max_order_value,float64,0.0,1860,0.0
3,recency_days,int64,0.0,741,0.0
0,n_orders,int64,0.0,17,0.0


In [18]:
# Show which features were dropped and why (audit)
if "model_base" in globals() and "candidate_features_approach2" in globals():
    target_col = [c for c in model_base.columns if c.startswith("y_")][0]
    dropped = [c for c in model_base.columns if c not in (["customer_id", target_col] + candidate_features_approach2)]
    print("Dropped features:", dropped)

Dropped features: ['first_order', 'last_order', 'person_id', 'store_id', 'territory_id']


In [19]:
# A.2 Insight Cell 1: Explain kept vs dropped features (screening audit)

if "model_base" not in globals():
    print("Run A.1 first to create model_base.")
else:
    df_fs = model_base.copy()

    meta = []
    for c in df_fs.columns:
        miss = df_fs[c].isna().mean()
        nun = df_fs[c].nunique(dropna=True)
        dtype = str(df_fs[c].dtype)
        meta.append((c, dtype, miss, nun))
    meta = pd.DataFrame(meta, columns=["feature", "dtype", "missing_rate", "n_unique"])
    meta["missing_%"] = (meta["missing_rate"] * 100).round(2)

    # Same screening logic as A.2
    keep_mask = (
        (meta["missing_rate"] <= 0.40) &
        ((meta["dtype"].str.contains("int|float")) | (meta["n_unique"].between(2, 50)) | (meta["feature"] == "customer_id"))
    )

    meta["decision"] = keep_mask.map({True: "KEEP", False: "DROP"})
    display(meta.sort_values(["decision", "missing_rate"], ascending=[True, False]))

,feature,dtype,missing_rate,n_unique,missing_%,decision
10,store_id,object,0.991609,138,99.16,DROP
9,person_id,object,0.429870,9232,42.99,DROP
11,territory_id,object,0.429870,4,42.99,DROP
2,first_order,datetime64[ns],0.000000,1124,0.00,DROP
3,last_order,datetime64[ns],0.000000,743,0.00,DROP
0,customer_id,object,0.000000,19119,0.00,KEEP
1,n_orders,int64,0.000000,17,0.00,KEEP
4,recency_days,int64,0.000000,741,0.00,KEEP
5,total_spend,float64,0.000000,5442,0.00,KEEP
6,avg_order_value,float64,0.000000,5510,0.00,KEEP


In [20]:
# A.2 Insight Cell 2: Visualise missingness vs cardinality for all features (excluding ID/target)

import altair as alt

if "model_base" not in globals():
    print("Run A.1 first to create model_base.")
else:
    df_fs = model_base.copy()

    # detect target column if present
    target_cols = [c for c in df_fs.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None

    cols_to_plot = [c for c in df_fs.columns if c not in ["customer_id", target_col]]

    if len(cols_to_plot) == 0:
        print("Nothing to plot: model_base only contains customer_id and/or target columns.")
    else:
        df_fs = df_fs[cols_to_plot].copy()

        meta = pd.DataFrame({
            "feature": cols_to_plot,
            "missing_%": (df_fs.isna().mean() * 100).round(2).values,
            "n_unique": [df_fs[c].nunique(dropna=True) for c in cols_to_plot],
            "dtype_raw": df_fs.dtypes.astype(str).values
        })
        meta["dtype"] = meta["dtype_raw"].apply(lambda d: "numeric" if ("int" in d or "float" in d) else "categorical")

        print("Meta rows (features plotted):", len(meta))
        display(meta.head())

        # IMPORTANT: disable Altair's max rows limit
        alt.data_transformers.disable_max_rows()

        # Try a renderer that works in most notebooks
        alt.renderers.enable("default")

        chart = alt.Chart(meta).mark_circle(size=140).encode(
            x=alt.X("missing_%:Q", title="Missing %"),
            y=alt.Y("n_unique:Q", title="Cardinality (# unique)", scale=alt.Scale(type="log")),
            color=alt.Color("dtype:N", title="Type"),
            tooltip=["feature:N", "dtype:N", "missing_%:Q", "n_unique:Q"]
        ).properties(
            title="A.2 Screening view: missingness vs cardinality (excluding ID/target)",
            width=750,
            height=350
        )

        chart

Meta rows (features plotted): 10


,feature,missing_%,n_unique,dtype_raw,dtype
0,n_orders,0.0,17,int64,numeric
1,first_order,0.0,1124,datetime64[ns],categorical
2,last_order,0.0,743,datetime64[ns],categorical
3,recency_days,0.0,741,int64,numeric
4,total_spend,0.0,5442,float64,numeric


In [21]:
feature_selection_2_insights = """
Approach 2 (Automated screening: missingness + cardinality)

Why this approach was chosen
- After creating the customer-level modelling base in Approach 1, this approach focuses on making the feature set practical and model-ready.
- In real datasets, many columns are unusable “as-is” because they are too sparse, behave like identifiers, or have extreme cardinality that encourages overfitting.
- Therefore, an automated screening step provides a transparent, repeatable way to reduce the feature space before encoding and scaling.

What was done (method)
- For each candidate predictor column (excluding customer_id and the target y_*), we computed:
  - missing_% (data completeness),
  - n_unique (cardinality / how many distinct values),
  - dtype (numeric vs categorical).
- We then applied simple filtering rules:
  - drop columns with missingness above a chosen threshold (e.g., 40%),
  - keep numeric columns with acceptable missingness (they are straightforward to impute and scale),
  - keep categorical columns only if their cardinality is within a reasonable range (e.g., 2 to 50 unique values),
  - exclude ID-like columns from being used as predictors (customer_id remains only for traceability/debugging and is dropped before modelling).

Results / outputs
- This approach outputs a shortlist (candidate_features_approach2) that:
  - has better coverage (less missingness),
  - avoids high-cardinality identifier-type features,
  - is easier to encode consistently across training/validation/testing splits.
- In practice, the retained features typically include:
  - behavioural aggregates (recency/frequency/monetary),
  - and a small number of stable segmentation attributes (e.g., territory/store/type) if they pass the screening rules.

Strengths and limitations
- Strengths:
  - Objective and repeatable (same rules give the same shortlist).
  - Reduces noise and improves model robustness by removing sparse/high-cardinality features early.
  - Provides clear justification for why certain features were removed (missingness/cardinality).
- Limitations:
  - Screening does not guarantee predictive usefulness; it only ensures the features are “usable”.
  - Thresholds are heuristic and may need tuning depending on dataset size and feature meaning.
  - Some high-cardinality columns may still be valuable after feature engineering (bucketing/target encoding), but they are intentionally excluded here for a clean baseline pipeline.

Conclusion
- Approach 2 complements Approach 1 by converting a broad feature set into a smaller, higher-quality, model-ready shortlist that can be reliably transformed and used for classification.
"""

In [22]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_2_insights', value=feature_selection_2_insights)

In [23]:
from IPython.display import display, HTML
import html

html_text = "<pre style='white-space: pre-wrap; word-wrap: break-word; font-size: 13px; line-height: 1.35;'>" \
            + html.escape(feature_selection_2_insights.strip()) + \
            "</pre>"

display(HTML(html_text))

### A.n Approach `Signal-based feature selection`

> You can add more cells related to other approaches in this section

In [24]:
sales_order_header_df.columns.tolist()
customer_df.columns.tolist()

['customer_id', 'person_id', 'store_id', 'territory_id', 'account_number']

In [25]:
# A.n Approach: Signal-based feature selection (simple, model-free)
# - Numeric: absolute correlation with target (point-biserial)
# - Categorical: mean target rate per category + weighted variance (how much the target changes across categories)
# This is stronger than "variance/entropy" because it uses y (still lightweight; not full ML training).

import numpy as np

if "model_base" not in globals():
    print("Run A.1 first to create model_base.")
else:
    # detect target
    target_cols = [c for c in model_base.columns if c.startswith("y_")]
    if len(target_cols) == 0:
        print("No target y_* column found. Create the label in A.1 first.")
    else:
        target_col = target_cols[0]
        df_an = model_base.copy()

        # ensure y is numeric 0/1
        df_an[target_col] = pd.to_numeric(df_an[target_col], errors="coerce")

        # candidates exclude ID + target + obvious datetime columns
        exclude = {"customer_id", target_col}
        candidates = [c for c in df_an.columns if c not in exclude]
        # drop datetime-like columns from scoring (keep them for feature eng if you want)
        candidates = [c for c in candidates if not np.issubdtype(df_an[c].dtype, np.datetime64)]

        scores = []

        for c in candidates:
            s = df_an[c]
            miss = s.isna().mean()
            nun = s.nunique(dropna=True)

            # skip constant columns
            if nun <= 1:
                continue

            if pd.api.types.is_numeric_dtype(s):
                tmp = df_an[[c, target_col]].dropna()
                if len(tmp) < 10:
                    continue
                corr = tmp[c].corr(tmp[target_col])  # Pearson corr with binary y
                score = abs(corr) if pd.notna(corr) else 0.0
                scores.append({
                    "feature": c,
                    "type": "numeric",
                    "missing_%": round(miss * 100, 2),
                    "n_unique": int(nun),
                    "signal_score": float(score),
                    "detail": f"abs(corr_with_{target_col})"
                })

            else:
                # categorical signal: how much target rate varies across categories
                tmp = df_an[[c, target_col]].dropna()
                if len(tmp) < 10:
                    continue

                # reduce to string labels
                tmp[c] = tmp[c].astype(str).str.strip().str.lower()

                # if too many categories, skip (or bucket top-k)
                if tmp[c].nunique() > 50:
                    continue

                grp = tmp.groupby(c)[target_col].mean()
                cnt = tmp.groupby(c)[target_col].size()

                # weighted variance of category target rates around global mean
                global_mean = tmp[target_col].mean()
                weights = cnt / cnt.sum()
                w_var = float((weights * (grp - global_mean) ** 2).sum())

                scores.append({
                    "feature": c,
                    "type": "categorical",
                    "missing_%": round(miss * 100, 2),
                    "n_unique": int(nun),
                    "signal_score": float(w_var),
                    "detail": "weighted_var(target_rate_by_category)"
                })

        scores_df = pd.DataFrame(scores).sort_values("signal_score", ascending=False)

        print("Top features by simple target-based signal score:")
        display(scores_df.head(25))

        # store top list for later
        candidate_features_an = scores_df.head(15)["feature"].tolist()
        print("candidate_features_an:", candidate_features_an)

Top features by simple target-based signal score:


,feature,type,missing_%,n_unique,signal_score,detail
0,n_orders,numeric,0.00,17,0.539954,abs(corr_with_y_reorder_within_90d)
2,total_spend,numeric,0.00,5442,0.265361,abs(corr_with_y_reorder_within_90d)
4,max_order_value,numeric,0.00,1860,0.261017,abs(corr_with_y_reorder_within_90d)
3,avg_order_value,numeric,0.00,5510,0.229339,abs(corr_with_y_reorder_within_90d)
1,recency_days,numeric,0.00,741,0.073891,abs(corr_with_y_reorder_within_90d)
5,territory_id,categorical,42.99,4,0.000985,weighted_var(target_rate_by_category)


candidate_features_an: ['n_orders', 'total_spend', 'max_order_value', 'avg_order_value', 'recency_days', 'territory_id']


In [26]:
# A.n Extra Cell 1: Explain/visualise top categorical signal drivers (if any)
# Shows category-wise target rates for the best categorical feature.

if "scores_df" not in globals() or len(scores_df) == 0:
    print("Run the A.n scoring cell first (scores_df not available).")
else:
    target_col = [c for c in model_base.columns if c.startswith("y_")][0]
    best_cat = scores_df[scores_df["type"] == "categorical"]["feature"].head(1).tolist()

    if len(best_cat) == 0:
        print("No categorical features passed the screening/scoring.")
    else:
        best_cat = best_cat[0]
        tmp = model_base[[best_cat, target_col]].dropna().copy()
        tmp[best_cat] = tmp[best_cat].astype(str).str.strip().str.lower()

        rate = tmp.groupby(best_cat)[target_col].mean().reset_index(name="target_rate")
        cnt = tmp.groupby(best_cat)[target_col].size().reset_index(name="count")
        out = rate.merge(cnt, on=best_cat).sort_values("count", ascending=False)

        print("Best categorical feature:", best_cat)
        display(out.head(20))

        alt.Chart(out).mark_bar().encode(
            x=alt.X("target_rate:Q", title=f"Mean {target_col} (target rate)"),
            y=alt.Y(f"{best_cat}:N", sort="-x", title=best_cat),
            tooltip=[best_cat, "target_rate", "count"]
        ).properties(title=f"Target rate by category: {best_cat}", width=750, height=350)

Best categorical feature: territory_id


,territory_id,target_rate,count
2,2ac923e9-3043-4f5e-bae0-ad28089bf187,0.120343,5127
3,e0e3bac4-790e-4617-84b3-be0c2bdb7070,0.136413,2771
0,0ad4c625-bb65-4376-8a41-0a65719b0db8,0.065567,2608
1,25d73ad2-9e13-410b-8b0d-5f26e2b9e4f2,0.060606,2607


In [27]:
# A.n Extra Cell 2: Numeric signal sanity check (scatter-ish using binned means)
# Avoids heavy modelling, but provides evidence the numeric feature relates to y.

if "scores_df" not in globals() or len(scores_df) == 0:
    print("Run the A.n scoring cell first (scores_df not available).")
else:
    target_col = [c for c in model_base.columns if c.startswith("y_")][0]
    best_num = scores_df[scores_df["type"] == "numeric"]["feature"].head(1).tolist()

    if len(best_num) == 0:
        print("No numeric features available for numeric signal plot.")
    else:
        best_num = best_num[0]
        tmp = model_base[[best_num, target_col]].dropna().copy()

        # bin into quantiles and compute target rate per bin
        tmp["bin"] = pd.qcut(tmp[best_num], q=10, duplicates="drop")
        b = tmp.groupby("bin").agg(
            bin_min=(best_num, "min"),
            bin_max=(best_num, "max"),
            target_rate=(target_col, "mean"),
            count=(target_col, "size")
        ).reset_index(drop=True)

        b["bin_label"] = b.apply(lambda r: f"{r['bin_min']:.2f}–{r['bin_max']:.2f}", axis=1)

        print("Best numeric feature:", best_num)
        display(b)

        alt.Chart(b).mark_line(point=True).encode(
            x=alt.X("bin_label:N", sort=None, title=f"{best_num} (quantile bins)"),
            y=alt.Y("target_rate:Q", title=f"Mean {target_col} (target rate)"),
            tooltip=["bin_label", "target_rate", "count"]
        ).properties(title=f"Target rate across {best_num} bins", width=750, height=300)

Best numeric feature: n_orders


,bin_min,bin_max,target_rate,count,bin_label
0,1,2,0.039399,20508,1.00–2.00
1,3,3,0.452174,1610,3.00–3.00
2,4,28,0.819728,882,4.00–28.00


In [28]:
# A.n Updated feature_selection_n_insights (paste into the text cell)
feature_selection_n_insights = """
Approach n (Target-based signal scoring: correlation + category target-rate variation)

Why this approach was chosen
- Approach 1 and 2 focus on business logic and data usability. This approach adds a lightweight, target-aware way to prioritise features without training a full model.
- It helps justify why certain features are more relevant to predicting the binary target (y_*), while still keeping the workflow simple and explainable.

What was done (method)
- Identified the target column (y_*).
- For numeric features:
  - computed correlation between the numeric feature and the binary target,
  - ranked features by absolute correlation magnitude (higher = stronger linear association).
- For categorical features (with reasonable cardinality):
  - computed the mean target rate for each category,
  - calculated a weighted variance of category target rates (higher = target rate changes meaningfully across categories).
- Excluded obvious identifiers (customer_id) and skipped constant or very sparse columns.

Results / outputs
- Produced a ranked table (scores_df) containing:
  - feature name,
  - type (numeric/categorical),
  - missingness,
  - cardinality,
  - a signal_score showing strength of association with the target.
- Generated a shortlist (candidate_features_an) of the top-scoring features to support the final feature set decision.

Strengths and limitations
- Strengths:
  - Uses the target to measure relevance without fitting a complex model.
  - Still interpretable: correlation and category-wise target rates are easy to explain.
  - Helps confirm whether behavioural features (recency/frequency/monetary) truly align with the label in this dataset.
- Limitations:
  - Correlation captures mainly linear relationships.
  - Categorical scoring depends on stable categories and can be noisy with small counts.
  - This does not replace full model-based feature importance; it is a lightweight prioritisation step.

Conclusion
- Approach n complements the earlier approaches by adding target-aware evidence for feature relevance, improving justification for the final selected feature set.
"""

In [29]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_n_insights', value=feature_selection_n_insights)

### A.z Final Selection of Features

In [30]:
# A.z Extra Cell 1: Build final feature set using a clear rule (intersection + must-have)
# - must_have: core RFM behaviour features (if present)
# - plus: features that passed Approach 2 screening
# - plus: top-k features from Approach n scoring (target-based signal)
# This gives a defensible final feature list.

if "model_base" not in globals():
    print("Run A.1 first to create model_base.")
else:
    # detect target
    target_cols = [c for c in model_base.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None
    if target_col is None:
        print("No target y_* found. Create the label in A.1 first.")
    else:
        # Must-have behavioural features (use only if they exist)
        must_have = [c for c in ["recency_days", "n_orders", "total_spend", "avg_order_value", "max_order_value"] if c in model_base.columns]

        # Approach 2 list (if created)
        a2 = []
        if "candidate_features_approach2" in globals():
            a2 = [c for c in candidate_features_approach2 if c in model_base.columns]

        # Approach n list (if created)
        an = []
        if "candidate_features_an" in globals():
            an = [c for c in candidate_features_an if c in model_base.columns]

        # Combine with priority: must_have + (A2 union top AN)
        final = must_have + a2 + an
        # remove IDs, target, and datetimes
        final = [c for c in final if c not in ["customer_id", target_col]]
        final = [c for c in final if not pd.api.types.is_datetime64_any_dtype(model_base[c])]
        # dedupe while preserving order
        features_list = list(dict.fromkeys(final))

        print("Target:", target_col)
        print("Final features_list (count):", len(features_list))
        print(features_list)

Target: y_reorder_within_90d
Final features_list (count): 6
['recency_days', 'n_orders', 'total_spend', 'avg_order_value', 'max_order_value', 'territory_id']


In [31]:
# A.z Extra Cell 2: Create modelling_df and verify grain (1 row per customer) + basic stats

if "model_base" not in globals() or "features_list" not in globals():
    print("Run A.1 and A.z Extra Cell 1 first.")
else:
    target_cols = [c for c in model_base.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None

    modelling_df = model_base[["customer_id"] + features_list + ([target_col] if target_col else [])].copy()

    # grain checks
    dup_customers = int(modelling_df["customer_id"].duplicated().sum())
    print("modelling_df shape:", modelling_df.shape)
    print("Duplicate customer_id rows (should be 0):", dup_customers)

    # missingness overview
    miss = (modelling_df.isna().mean() * 100).round(2).sort_values(ascending=False).reset_index()
    miss.columns = ["feature", "missing_%"]
    display(miss.head(20))

    display(modelling_df.head())

modelling_df shape: (23000, 8)
Duplicate customer_id rows (should be 0): 3881


,feature,missing_%
0,territory_id,42.99
1,customer_id,0.00
2,recency_days,0.00
3,n_orders,0.00
4,total_spend,0.00
5,avg_order_value,0.00
6,max_order_value,0.00
7,y_reorder_within_90d,0.00


,customer_id,recency_days,n_orders,total_spend,avg_order_value,max_order_value,territory_id,y_reorder_within_90d
0,00027a37-6f01-4a8f-bd81-23f2a6e7f525,32,2,6673.4266,3336.7133,3953.9884,NaN,0
1,0002bd5d-7aa8-403e-aa8c-d7134d92da55,32,1,937.5594,937.5594,937.5594,NaN,0
2,000421bb-5918-41c8-b95f-3bc0887876dd,296,1,35.6694,35.6694,35.6694,NaN,0
3,000a3e10-bb31-43db-adf5-a4450ea0cc7e,3,1,44.1779,44.1779,44.1779,NaN,0
4,000bc388-bef0-46fd-8f64-cd76f2350ef1,163,1,8.0444,8.0444,8.0444,NaN,0


In [32]:
# A.z Extra Cell 3: Feature types summary + cardinality warnings
# This helps justify that your final features are usable for encoding and modelling.

if "modelling_df" not in globals():
    print("Create modelling_df first (A.z Extra Cell 2).")
else:
    target_cols = [c for c in modelling_df.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None

    predictors = [c for c in modelling_df.columns if c not in ["customer_id", target_col]]
    info = []
    for c in predictors:
        dtype = str(modelling_df[c].dtype)
        nun = int(modelling_df[c].nunique(dropna=True))
        miss = float(modelling_df[c].isna().mean())
        info.append((c, dtype, nun, round(miss * 100, 2)))

    info = pd.DataFrame(info, columns=["feature", "dtype", "n_unique", "missing_%"]).sort_values(["dtype", "n_unique"], ascending=[True, False])
    display(info)

    # warn about high-cardinality categoricals
    high_card = info[(~info["dtype"].str.contains("int|float")) & (info["n_unique"] > 50)]
    if len(high_card) > 0:
        print("WARNING: High-cardinality categorical predictors found (consider dropping/bucketing):")
        display(high_card)
    else:
        print("No high-cardinality categorical predictors in final set (good).")

,feature,dtype,n_unique,missing_%
3,avg_order_value,float64,5510,0.00
2,total_spend,float64,5442,0.00
4,max_order_value,float64,1860,0.00
0,recency_days,int64,741,0.00
1,n_orders,int64,17,0.00
5,territory_id,object,4,42.99


No high-cardinality categorical predictors in final set (good).


In [33]:
# A.z Extra Cell 4: Quick leakage sanity check (ensure we did not include target-like columns)
# Helps you prove you are not accidentally using y or future info as a predictor.

if "modelling_df" not in globals():
    print("Create modelling_df first.")
else:
    bad = [c for c in modelling_df.columns if "y_" in c and c != [x for x in modelling_df.columns if x.startswith("y_")][0]]
    suspicious = [c for c in modelling_df.columns if any(k in c.lower() for k in ["target", "label", "next", "future"])]
    print("Other y_* columns present (should be none besides target):", bad)
    print("Suspicious leakage-like column names:", suspicious)

Other y_* columns present (should be none besides target): ['recency_days', 'territory_id']
Suspicious leakage-like column names: []


In [34]:
# A.z Extra Cell 5: Final audit of selected features (quality gate)

if "model_base" not in globals() or "features_list" not in globals():
    print("Run A.1 and create features_list first.")
else:
    # detect target
    target_cols = [c for c in model_base.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None

    audit_df = model_base[features_list].copy()

    audit = []
    for c in audit_df.columns:
        miss = audit_df[c].isna().mean()
        nun = audit_df[c].nunique(dropna=True)
        is_constant = (nun <= 1)
        audit.append({
            "feature": c,
            "dtype": str(audit_df[c].dtype),
            "missing_%": round(miss * 100, 2),
            "n_unique": int(nun),
            "constant_or_single_value": bool(is_constant),
        })

    audit = pd.DataFrame(audit).sort_values(["constant_or_single_value", "missing_%"], ascending=[False, False])
    display(audit)

    # Recommend drops
    to_drop = audit.loc[(audit["constant_or_single_value"]) | (audit["missing_%"] > 40), "feature"].tolist()
    print("Suggested features to drop (if any):", to_drop)

,feature,dtype,missing_%,n_unique,constant_or_single_value
5,territory_id,object,42.99,4,False
0,recency_days,int64,0.00,741,False
1,n_orders,int64,0.00,17,False
2,total_spend,float64,0.00,5442,False
3,avg_order_value,float64,0.00,5510,False
4,max_order_value,float64,0.00,1860,False


Suggested features to drop (if any): ['territory_id']


In [35]:
# A.z Extra Cell 6: Optional automatic cleanup of features_list based on audit rules

if "model_base" not in globals() or "features_list" not in globals():
    print("Run A.1 and create features_list first.")
else:
    Xcand = model_base[features_list].copy()

    good = []
    dropped = []
    for c in features_list:
        miss = Xcand[c].isna().mean()
        nun = Xcand[c].nunique(dropna=True)
        if (miss > 0.40) or (nun <= 1):
            dropped.append(c)
        else:
            good.append(c)

    features_list = good  # update in-place
    print("Dropped from features_list:", dropped)
    print("Updated features_list (count):", len(features_list))
    print(features_list)

Dropped from features_list: ['territory_id']
Updated features_list (count): 5
['recency_days', 'n_orders', 'total_spend', 'avg_order_value', 'max_order_value']


In [36]:
# A.z Extra Cell 7: Stability check (early vs late split) - drift proxy
# Uses last_order date to define early/late groups and compares numeric means.

import numpy as np

if "model_base" not in globals() or "features_list" not in globals():
    print("Run A.1 and A.z first.")
else:
    if "last_order" not in model_base.columns:
        print("last_order not found in model_base; cannot run stability check.")
    else:
        tmp = model_base[["last_order"] + features_list].copy()
        tmp["last_order"] = pd.to_datetime(tmp["last_order"], errors="coerce")
        tmp = tmp.dropna(subset=["last_order"])

        # early vs late split at median last_order
        cut = tmp["last_order"].median()
        tmp["period"] = np.where(tmp["last_order"] <= cut, "early", "late")

        num_feats = [c for c in features_list if pd.api.types.is_numeric_dtype(tmp[c])]
        if len(num_feats) == 0:
            print("No numeric features to compare.")
        else:
            early = tmp[tmp["period"] == "early"][num_feats]
            late = tmp[tmp["period"] == "late"][num_feats]

            comp = pd.DataFrame({
                "feature": num_feats,
                "early_mean": early.mean().values,
                "late_mean": late.mean().values,
            })
            comp["mean_diff"] = comp["late_mean"] - comp["early_mean"]
            comp["pct_diff_vs_early"] = (comp["mean_diff"] / comp["early_mean"].replace(0, np.nan) * 100).round(2)

            display(comp.sort_values("pct_diff_vs_early", ascending=False).head(20))
            print("Note: Large pct_diff may indicate temporal drift; consider time-based split and/or regularisation.")

,feature,early_mean,late_mean,mean_diff,pct_diff_vs_early
2,total_spend,3102.213726,8789.728223,5687.514497,183.34
4,max_order_value,1655.728909,2609.108871,953.379963,57.58
3,avg_order_value,1332.096154,1899.417076,567.320921,42.59
1,n_orders,1.419782,1.873908,0.454126,31.99
0,recency_days,288.810402,84.710678,-204.099723,-70.67


Note: Large pct_diff may indicate temporal drift; consider time-based split and/or regularisation.


In [37]:
# A.z Extra Cell 8: Leakage guard rail (drop columns that look like future/label information)

if "model_base" not in globals() or "features_list" not in globals():
    print("Run A.1 and A.z first.")
else:
    suspicious_keywords = ["target", "label", "next", "future", "reorder", "churn", "y_"]
    suspicious = [c for c in features_list if any(k in c.lower() for k in suspicious_keywords)]

    if len(suspicious) > 0:
        print("Suspicious columns detected (will be removed):", suspicious)
        features_list = [c for c in features_list if c not in suspicious]
    else:
        print("No suspicious leakage-like predictors detected.")

    print("Final features_list after leakage guard (count):", len(features_list))
    print(features_list)

Suspicious columns detected (will be removed): ['recency_days']
Final features_list after leakage guard (count): 4
['n_orders', 'total_spend', 'avg_order_value', 'max_order_value']


In [38]:
feature_selection_explanations = """
Final feature selection strategy (expanded explanation)

Objective (what we are predicting)
- This project is a classification task (Student A). The goal is to predict a customer outcome (target y_*) such as whether a customer will reorder within a chosen future time window.
- Therefore, the selected predictors must be:
  1) available at the time of prediction (to avoid leakage),
  2) stable and usable (low missingness, not just IDs),
  3) relevant to customer purchasing behaviour.

Step 1 — Start with business-meaningful baseline predictors (RFM logic)
- We first prioritised behavioural features derived from order history because they are widely validated in customer analytics:
  - recency_days: measures how recently the customer purchased. Recent customers are typically more likely to purchase again.
  - n_orders: measures purchase frequency/engagement. Customers who have purchased more often tend to be more loyal.
  - total_spend: measures overall customer value. Higher-value customers often exhibit different repurchase patterns.
  - avg_order_value: captures typical basket size/spend intensity per order.
  - max_order_value: captures “peak” purchasing behaviour which can differentiate occasional high spenders.
- These features are interpretable and easy to justify in the report.

Step 2 — Apply quality screening (Approach 2) to ensure features are usable
- Even if a feature is interesting, it may not be usable if:
  - it has high missingness (too many blanks, requiring heavy imputation and introducing noise),
  - it has extremely high cardinality (e.g., unique codes/IDs) that causes the model to memorise rather than generalise,
  - or it has little practical variation.
- Approach 2 enforces objective rules (missingness threshold + cardinality limits), producing a shortlist of predictors that are realistic to encode and scale across train/val/test.

Step 3 — Add target-aware prioritisation (Approach n) without heavy modelling
- After filtering for usability, we still want evidence that features relate to the target.
- Approach n uses lightweight signal measures:
  - for numeric features: correlation with the binary target,
  - for categorical features: how much the target rate changes across categories (target-rate variation).
- This step does not train a full ML model; it provides quick, explainable evidence to prioritise features that show actual association with y_*.

Final selection outcome (why this is defensible)
- The final features_list is the result of combining:
  - strong business-driven predictors (RFM),
  - objective data quality filters (missingness/cardinality),
  - and target-aware relevance checks (signal scoring).
- This makes the selection:
  - Relevant: features reflect customer behaviour and segmentation.
  - Usable: manageable missingness and encoding complexity.
  - Low leakage: avoids using the target itself or future-only information.
  - Explainable: each feature category (RFM / screened / signal-supported) has a clear reason for inclusion.

Note
- customer_id is retained only for traceability and join/debugging and should be removed before training the final model.
"""

In [39]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## B. Data Cleaning

### B.1 Fixing `missing values`

In [40]:
# B.1 Fixing: missing values in modelling_df

if "modelling_df" not in globals():
    print("Run A.z to create modelling_df first.")
else:
    before_missing = (modelling_df.isna().mean() * 100).round(2).sort_values(ascending=False)
    print("Missing % before cleaning:")
    display(before_missing)

    cleaned_df = modelling_df.copy()

    # Numeric: fill with median
    num_cols = cleaned_df.select_dtypes(include="number").columns.tolist()
    for c in num_cols:
        cleaned_df[c] = cleaned_df[c].fillna(cleaned_df[c].median())

    # Categorical: fill with 'Unknown'
    cat_cols = [c for c in cleaned_df.columns if c not in num_cols]
    for c in cat_cols:
        cleaned_df[c] = cleaned_df[c].fillna("Unknown").astype(str)

    after_missing = (cleaned_df.isna().mean() * 100).round(2).sort_values(ascending=False)
    print("Missing % after cleaning:")
    display(after_missing)

Missing % before cleaning:


territory_id            42.99
customer_id              0.00
recency_days             0.00
n_orders                 0.00
total_spend              0.00
avg_order_value          0.00
max_order_value          0.00
y_reorder_within_90d     0.00
dtype: float64

Missing % after cleaning:


customer_id             0.0
recency_days            0.0
n_orders                0.0
total_spend             0.0
avg_order_value         0.0
max_order_value         0.0
territory_id            0.0
y_reorder_within_90d    0.0
dtype: float64

In [41]:
# Quick sanity check
if "cleaned_df" in globals():
    display(cleaned_df.head())
    print("cleaned_df shape:", cleaned_df.shape)

,customer_id,recency_days,n_orders,total_spend,avg_order_value,max_order_value,territory_id,y_reorder_within_90d
0,00027a37-6f01-4a8f-bd81-23f2a6e7f525,32,2,6673.4266,3336.7133,3953.9884,Unknown,0
1,0002bd5d-7aa8-403e-aa8c-d7134d92da55,32,1,937.5594,937.5594,937.5594,Unknown,0
2,000421bb-5918-41c8-b95f-3bc0887876dd,296,1,35.6694,35.6694,35.6694,Unknown,0
3,000a3e10-bb31-43db-adf5-a4450ea0cc7e,3,1,44.1779,44.1779,44.1779,Unknown,0
4,000bc388-bef0-46fd-8f64-cd76f2350ef1,163,1,8.0444,8.0444,8.0444,Unknown,0


cleaned_df shape: (23000, 8)


In [42]:
# B.1 Coding Cell 1: Missingness audit (before cleaning)
# Goal: clearly identify which features have missing values and how severe it is.

if "modelling_df" not in globals():
    print("Run A.z first to create modelling_df.")
else:
    miss = pd.DataFrame({
        "feature": modelling_df.columns,
        "dtype": modelling_df.dtypes.astype(str).values,
        "missing_count": modelling_df.isna().sum().values,
        "missing_%": (modelling_df.isna().mean() * 100).round(2).values,
        "n_unique": [modelling_df[c].nunique(dropna=True) for c in modelling_df.columns],
    }).sort_values(["missing_%", "missing_count"], ascending=[False, False])

    print("Missingness audit (top 25 by missing %):")
    display(miss.head(25))

Missingness audit (top 25 by missing %):


,feature,dtype,missing_count,missing_%,n_unique
6,territory_id,object,9887,42.99,4
0,customer_id,object,0,0.00,19119
1,recency_days,int64,0,0.00,741
2,n_orders,int64,0,0.00,17
3,total_spend,float64,0,0.00,5442
4,avg_order_value,float64,0,0.00,5510
5,max_order_value,float64,0,0.00,1860
7,y_reorder_within_90d,int64,0,0.00,2


In [43]:
# B.1 Coding Cell 2: Define the cleaning plan (what will be imputed and how)
# Goal: make the logic explicit and reproducible.

import numpy as np

if "modelling_df" not in globals():
    print("Run A.z first to create modelling_df.")
else:
    df_b1 = modelling_df.copy()

    # detect target if present
    target_cols = [c for c in df_b1.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None

    id_cols = ["customer_id"]
    protected_cols = [c for c in id_cols + ([target_col] if target_col else []) if c in df_b1.columns]

    # identify columns to impute (exclude id/target)
    cols_to_impute = [c for c in df_b1.columns if c not in protected_cols]

    numeric_cols = [c for c in cols_to_impute if pd.api.types.is_numeric_dtype(df_b1[c])]
    categorical_cols = [c for c in cols_to_impute if c not in numeric_cols]

    print("Protected (not imputed):", protected_cols)
    print("Numeric columns to impute (median):", numeric_cols)
    print("Categorical columns to impute ('Unknown'):", categorical_cols)

Protected (not imputed): ['customer_id', 'y_reorder_within_90d']
Numeric columns to impute (median): ['recency_days', 'n_orders', 'total_spend', 'avg_order_value', 'max_order_value']
Categorical columns to impute ('Unknown'): ['territory_id']


In [44]:
# B.1 Coding Cell 3: Impute missing values (median for numeric, 'Unknown' for categorical)
# Goal: produce a cleaned dataset for later steps.

if "modelling_df" not in globals():
    print("Run A.z first to create modelling_df.")
else:
    df_b1_clean = modelling_df.copy()

    # detect target if present
    target_cols = [c for c in df_b1_clean.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None

    protected_cols = [c for c in ["customer_id", target_col] if c and c in df_b1_clean.columns]

    # numeric median imputation
    for c in df_b1_clean.columns:
        if c in protected_cols:
            continue
        if pd.api.types.is_numeric_dtype(df_b1_clean[c]):
            med = df_b1_clean[c].median()
            df_b1_clean[c] = df_b1_clean[c].fillna(med)
        else:
            df_b1_clean[c] = df_b1_clean[c].fillna("Unknown").astype(str)

    print("B.1 imputation complete.")
    display(df_b1_clean.head())

B.1 imputation complete.


,customer_id,recency_days,n_orders,total_spend,avg_order_value,max_order_value,territory_id,y_reorder_within_90d
0,00027a37-6f01-4a8f-bd81-23f2a6e7f525,32,2,6673.4266,3336.7133,3953.9884,Unknown,0
1,0002bd5d-7aa8-403e-aa8c-d7134d92da55,32,1,937.5594,937.5594,937.5594,Unknown,0
2,000421bb-5918-41c8-b95f-3bc0887876dd,296,1,35.6694,35.6694,35.6694,Unknown,0
3,000a3e10-bb31-43db-adf5-a4450ea0cc7e,3,1,44.1779,44.1779,44.1779,Unknown,0
4,000bc388-bef0-46fd-8f64-cd76f2350ef1,163,1,8.0444,8.0444,8.0444,Unknown,0


In [45]:
# B.1 Coding Cell 4: Missingness audit (after cleaning) + sanity checks
# Goal: prove the issue was fixed and the dataset is still valid.

if "df_b1_clean" not in globals():
    print("Run B.1 Coding Cell 3 first.")
else:
    after = pd.DataFrame({
        "feature": df_b1_clean.columns,
        "missing_count": df_b1_clean.isna().sum().values,
        "missing_%": (df_b1_clean.isna().mean() * 100).round(2).values,
    }).sort_values(["missing_%", "missing_count"], ascending=[False, False])

    print("Missingness after cleaning (should be 0% for most/all predictors):")
    display(after.head(25))

    # sanity checks
    print("Total null cells:", int(df_b1_clean.isna().sum().sum()))
    if "customer_id" in df_b1_clean.columns:
        print("Duplicate customer_id rows:", int(df_b1_clean["customer_id"].duplicated().sum()))

Missingness after cleaning (should be 0% for most/all predictors):


,feature,missing_count,missing_%
0,customer_id,0,0.0
1,recency_days,0,0.0
2,n_orders,0,0.0
3,total_spend,0,0.0
4,avg_order_value,0,0.0
5,max_order_value,0,0.0
6,territory_id,0,0.0
7,y_reorder_within_90d,0,0.0


Total null cells: 0
Duplicate customer_id rows: 3881


In [46]:
# B.1 Coding Cell 5: Add missing-indicator flags for features that originally had missingness
# This can improve model performance because 'missingness' may be informative.

if "modelling_df" not in globals() or "df_b1_clean" not in globals():
    print("Run B.1 Coding Cell 3 first.")
else:
    # create flags only for columns that had missing values originally
    missing_cols = [c for c in modelling_df.columns if modelling_df[c].isna().any() and c not in ["customer_id"] and not c.startswith("y_")]

    df_b1_clean_flags = df_b1_clean.copy()
    for c in missing_cols:
        df_b1_clean_flags[f"{c}__was_missing"] = modelling_df[c].isna().astype(int)

    print("Added missing indicator columns:", [f"{c}__was_missing" for c in missing_cols])
    display(df_b1_clean_flags.head())

    # Use this as the cleaned dataset going forward:
    cleaned_df = df_b1_clean_flags
    print("cleaned_df shape:", cleaned_df.shape)

Added missing indicator columns: ['territory_id__was_missing']


,customer_id,recency_days,n_orders,total_spend,avg_order_value,max_order_value,territory_id,y_reorder_within_90d,territory_id__was_missing
0,00027a37-6f01-4a8f-bd81-23f2a6e7f525,32,2,6673.4266,3336.7133,3953.9884,Unknown,0,1
1,0002bd5d-7aa8-403e-aa8c-d7134d92da55,32,1,937.5594,937.5594,937.5594,Unknown,0,1
2,000421bb-5918-41c8-b95f-3bc0887876dd,296,1,35.6694,35.6694,35.6694,Unknown,0,1
3,000a3e10-bb31-43db-adf5-a4450ea0cc7e,3,1,44.1779,44.1779,44.1779,Unknown,0,1
4,000bc388-bef0-46fd-8f64-cd76f2350ef1,163,1,8.0444,8.0444,8.0444,Unknown,0,1


cleaned_df shape: (23000, 9)


In [47]:
data_cleaning_1_explanations = """
B.1 Fixing issue: Missing values (nulls / blanks) in the modelling dataset

What the issue is
- After creating the customer-level modelling table, some features can contain missing values.
- Missingness can occur for practical reasons, for example:
  - a customer has limited order history, so monetary or behavioural aggregates may be undefined,
  - some customer attributes (e.g., territory/store/person fields) are not recorded for every customer,
  - parsing issues (e.g., invalid dates) can produce NaT which then affects derived features such as recency_days.

Why it is important to fix
- Most machine learning algorithms and many preprocessing steps (scaling, encoding, correlation checks) cannot handle NaN values directly.
- If missing values are not handled, model training may fail, or rows with missing data may be dropped automatically, reducing the dataset size and potentially biasing the training sample.
- In addition, missingness itself can be systematic (not random). For example, new customers may have missing monetary history; removing them can bias the model against new-customer behaviour.

Impact on modelling and evaluation (if not fixed)
- Training impact:
  - Many models will error or implicitly drop rows with NaNs, leading to inconsistent training sets.
  - Features with high missingness can introduce noise and unstable decision boundaries.
- Evaluation impact:
  - If missingness differs across training/validation/testing splits, performance metrics become unreliable and not comparable.
  - The model may appear to perform well simply because rows with missing values were excluded (selection bias).
- Operational impact:
  - In a real deployment, incoming data will often have missing fields; the pipeline must be able to produce predictions even when some inputs are absent.

How the issue is fixed (and why the chosen strategy is reasonable)
- Numeric features are imputed using the median:
  - Median is robust to outliers (common in spend-related variables).
  - Keeps the feature on a realistic scale without being overly influenced by extreme values.
- Categorical features are imputed with an explicit 'Unknown' category:
  - Preserves rows instead of dropping them.
  - Allows the model to learn whether “unknown/missing category” has predictive meaning.
- This approach is simple, reproducible, and appropriate for a baseline preparation pipeline.

What to watch out for (limitations)
- Imputation can reduce variance and slightly weaken signal, especially if a feature is heavily missing.
- If a feature has extremely high missingness, it may be better to drop it (handled in A.2/A.z screening).
- For production-grade work, it can be useful to add a “missing indicator” feature per column, but for this assignment the current strategy is sufficient and defensible.

Conclusion
- Fixing missing values ensures the dataset is usable for modelling, prevents training failures, reduces bias from dropping data, and makes the pipeline robust to incomplete real-world inputs.
"""

In [48]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_1_explanations', value=data_cleaning_1_explanations)

### B.2 Fixing `Inconsistent categorical formatting (case/whitespace/rare categories)`

In [49]:
# B.2 Coding Cell 1: Identify and quantify inconsistent categorical formatting
# Issue targeted: messy text categories (extra spaces, inconsistent casing, "nan"/empty strings)

if "cleaned_df" not in globals():
    print("Run B.1 first to create cleaned_df.")
else:
    df_b2 = cleaned_df.copy()

    # Categorical candidates (object/string/category) excluding ID/target
    target_cols = [c for c in df_b2.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None

    cat_cols = df_b2.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    cat_cols = [c for c in cat_cols if c not in ["customer_id", target_col]]

    print("Categorical columns detected for standardisation:", cat_cols)

    # Show top values and whether whitespace/case variants exist (simple indicators)
    for c in cat_cols:
        s = df_b2[c].astype(str)

        # indicators
        has_leading_trailing = (s != s.str.strip()).any()
        has_upper = (s != s.str.lower()).any()
        has_multiple_spaces = s.str.contains(r"\s{2,}", regex=True).any()
        has_empty = (s.str.strip() == "").any()
        has_str_nan = s.str.lower().isin(["nan", "none", "null"]).any()

        print(f"\nColumn: {c}")
        print("  leading/trailing spaces:", bool(has_leading_trailing))
        print("  mixed casing present:", bool(has_upper))
        print("  multiple spaces present:", bool(has_multiple_spaces))
        print("  empty strings present:", bool(has_empty))
        print("  string 'nan/none/null' present:", bool(has_str_nan))

        display(df_b2[c].value_counts(dropna=False).head(10).reset_index().rename(columns={"index": c, c: "count"}))

Categorical columns detected for standardisation: ['territory_id']

Column: territory_id
  leading/trailing spaces: False
  mixed casing present: True
  multiple spaces present: False
  empty strings present: False
  string 'nan/none/null' present: False


,count,count
0,Unknown,9887
1,2ac923e9-3043-4f5e-bae0-ad28089bf187,5127
2,e0e3bac4-790e-4617-84b3-be0c2bdb7070,2771
3,0ad4c625-bb65-4376-8a41-0a65719b0db8,2608
4,25d73ad2-9e13-410b-8b0d-5f26e2b9e4f2,2607


In [50]:
# B.2 Coding Cell 2: Standardise categorical columns (trim, collapse spaces, consistent casing)
# Output: cleaned_df_b2

import re

if "cleaned_df" not in globals():
    print("Run B.1 first to create cleaned_df.")
else:
    df_b2 = cleaned_df.copy()

    target_cols = [c for c in df_b2.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None

    cat_cols = df_b2.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    cat_cols = [c for c in cat_cols if c not in ["customer_id", target_col]]

    def normalise_text(x):
        if pd.isna(x):
            return "unknown"
        x = str(x)
        x = x.strip()
        x = re.sub(r"\s+", " ", x)  # collapse multiple whitespace
        x = x.lower()
        # unify common missing tokens
        if x in ["", "nan", "none", "null", "na", "n/a"]:
            return "unknown"
        return x

    for c in cat_cols:
        df_b2[c] = df_b2[c].apply(normalise_text)

    cleaned_df_b2 = df_b2
    print("B.2 standardisation complete.")
    display(cleaned_df_b2.head())

B.2 standardisation complete.


,customer_id,recency_days,n_orders,total_spend,avg_order_value,max_order_value,territory_id,y_reorder_within_90d,territory_id__was_missing
0,00027a37-6f01-4a8f-bd81-23f2a6e7f525,32,2,6673.4266,3336.7133,3953.9884,unknown,0,1
1,0002bd5d-7aa8-403e-aa8c-d7134d92da55,32,1,937.5594,937.5594,937.5594,unknown,0,1
2,000421bb-5918-41c8-b95f-3bc0887876dd,296,1,35.6694,35.6694,35.6694,unknown,0,1
3,000a3e10-bb31-43db-adf5-a4450ea0cc7e,3,1,44.1779,44.1779,44.1779,unknown,0,1
4,000bc388-bef0-46fd-8f64-cd76f2350ef1,163,1,8.0444,8.0444,8.0444,unknown,0,1


In [51]:
# B.2 Coding Cell 3: Verify that standardisation reduced category fragmentation
# Shows before vs after cardinality and top categories.

if "cleaned_df_b2" not in globals():
    print("Run B.2 Coding Cell 2 first to create cleaned_df_b2.")
else:
    df_before = cleaned_df.copy()
    df_after = cleaned_df_b2.copy()

    target_cols = [c for c in df_after.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None

    cat_cols = df_after.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    cat_cols = [c for c in cat_cols if c not in ["customer_id", target_col]]

    comp = []
    for c in cat_cols:
        comp.append({
            "feature": c,
            "unique_before": int(df_before[c].nunique(dropna=True)),
            "unique_after": int(df_after[c].nunique(dropna=True)),
        })

    comp = pd.DataFrame(comp).sort_values("unique_after", ascending=False)
    print("Cardinality comparison (before vs after):")
    display(comp)

    # show top categories after cleaning for a quick sanity check
    for c in cat_cols:
        print(f"\nTop categories after standardisation: {c}")
        display(df_after[c].value_counts().head(10).reset_index().rename(columns={"index": c, c: "count"}))

Cardinality comparison (before vs after):


,feature,unique_before,unique_after
0,territory_id,5,5



Top categories after standardisation: territory_id


,count,count
0,unknown,9887
1,2ac923e9-3043-4f5e-bae0-ad28089bf187,5127
2,e0e3bac4-790e-4617-84b3-be0c2bdb7070,2771
3,0ad4c625-bb65-4376-8a41-0a65719b0db8,2608
4,25d73ad2-9e13-410b-8b0d-5f26e2b9e4f2,2607


In [52]:
# B.2 Coding Cell 4 (Optional HD): Consolidate rare categories into "__other__"
# Helps avoid sparse one-hot columns later.

if "cleaned_df_b2" not in globals():
    print("Run B.2 Coding Cell 2 first.")
else:
    df_b2 = cleaned_df_b2.copy()

    target_cols = [c for c in df_b2.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None

    cat_cols = df_b2.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    cat_cols = [c for c in cat_cols if c not in ["customer_id", target_col]]

    MIN_COUNT = 20  # adjust based on dataset size; 10–50 are common

    for c in cat_cols:
        vc = df_b2[c].value_counts()
        rare = vc[vc < MIN_COUNT].index
        if len(rare) > 0:
            df_b2[c] = df_b2[c].where(~df_b2[c].isin(rare), "__other__")

    cleaned_df = df_b2  # overwrite cleaned_df for next steps
    print("Rare-category consolidation complete. cleaned_df updated.")
    display(cleaned_df.head())

Rare-category consolidation complete. cleaned_df updated.


,customer_id,recency_days,n_orders,total_spend,avg_order_value,max_order_value,territory_id,y_reorder_within_90d,territory_id__was_missing
0,00027a37-6f01-4a8f-bd81-23f2a6e7f525,32,2,6673.4266,3336.7133,3953.9884,unknown,0,1
1,0002bd5d-7aa8-403e-aa8c-d7134d92da55,32,1,937.5594,937.5594,937.5594,unknown,0,1
2,000421bb-5918-41c8-b95f-3bc0887876dd,296,1,35.6694,35.6694,35.6694,unknown,0,1
3,000a3e10-bb31-43db-adf5-a4450ea0cc7e,3,1,44.1779,44.1779,44.1779,unknown,0,1
4,000bc388-bef0-46fd-8f64-cd76f2350ef1,163,1,8.0444,8.0444,8.0444,unknown,0,1


In [53]:
data_cleaning_2_explanations = """
B.2 Fixing issue: Inconsistent categorical/text formatting (case, whitespace, invalid tokens, and rare categories)

What the issue is
- Several predictors in the modelling dataset are categorical (stored as text/object), for example customer segmentation fields such as territory/store/type/person-related categories.
- In raw CSV exports, categorical columns frequently contain inconsistent formatting, including:
  - leading/trailing spaces (e.g., "north " vs "north"),
  - inconsistent casing (e.g., "NORTH", "North", "north"),
  - multiple internal spaces (e.g., "new  york"),
  - placeholder strings for missing values (e.g., "", "nan", "null", "none"),
  - and very rare categories (categories that appear only a few times).

Why it is important to fix
1) Prevents artificial category fragmentation
- Without standardisation, the model treats formatting variants as different categories.
- This increases the number of categories and creates redundant one-hot encoded columns that represent the same concept.

2) Improves model generalisation and reduces overfitting risk
- High-cardinality categoricals lead to many sparse dummy variables.
- Sparse one-hot features make it easier for a model to memorise noise (especially if a category appears only a small number of times), which reduces performance on validation/testing data.

3) Ensures consistent train/validation/testing transformations
- If category labels are messy, the set of categories observed in training can differ from validation/testing even when the underlying concept is the same (e.g., "North" vs "north").
- This causes inconsistent encoding and can lead to unseen-category problems during evaluation or deployment.

4) Increases interpretability and reporting quality
- Clean categories are easier to explain in the final report and make model outputs more trustworthy (e.g., feature importance for "territory=north" is clearer than having multiple near-duplicate categories).

How the issue is fixed (and why the approach is reasonable)
- Standardisation steps applied to each categorical column:
  - convert to string, strip whitespace, collapse repeated spaces,
  - apply consistent casing (lowercase) to unify labels,
  - convert empty strings and text missing tokens ("nan", "null", "none") into a single explicit category "unknown".
- Optional improvement for modelling stability:
  - consolidate rare categories (e.g., frequency < MIN_COUNT) into "__other__" to reduce sparsity.
  - This retains information that the category is uncommon without creating many nearly-empty dummy columns.

Impacts after fixing
- Fewer unique values per categorical feature (reduced cardinality).
- More stable one-hot encoding across splits.
- Lower noise and better generalisation potential for downstream models.
- Cleaner feature space that supports reproducible modelling and clearer interpretation.

Conclusion
- Fixing categorical formatting is essential for building a robust, reproducible modelling pipeline. It reduces unnecessary dimensionality, prevents encoding inconsistencies across splits, and improves the reliability of model training and evaluation.
"""

In [54]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_2_explanations', value=data_cleaning_2_explanations)

### B.3 Fixing "\<describe_issue_here\>"

In [55]:
# B.3 Coding Cell 1: Identify numeric features and audit outliers (before capping)

import numpy as np

if "cleaned_df" not in globals():
    print("Run B.1 and B.2 first to create cleaned_df.")
else:
    df_b3 = cleaned_df.copy()

    # exclude ID + target from outlier processing
    target_cols = [c for c in df_b3.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None

    protected = ["customer_id"] + ([target_col] if target_col else [])
    num_cols = df_b3.select_dtypes(include="number").columns.tolist()
    num_cols = [c for c in num_cols if c not in protected]

    # focus on likely skewed monetary/behavioural columns first (still numeric)
    focus_cols = [c for c in num_cols if any(k in c.lower() for k in ["spend", "value", "amount", "total", "due", "subtotal", "price"])]
    # always include core RFM numeric columns if present
    for c in ["recency_days", "n_orders", "total_spend", "avg_order_value", "max_order_value"]:
        if c in num_cols and c not in focus_cols:
            focus_cols.append(c)

    if len(focus_cols) == 0:
        focus_cols = num_cols  # fallback to all numeric

    def outlier_summary(df, cols):
        rows = []
        for c in cols:
            s = df[c].dropna()
            if len(s) == 0:
                continue
            q01, q50, q99 = s.quantile([0.01, 0.50, 0.99]).tolist()
            iqr = s.quantile(0.75) - s.quantile(0.25)
            rows.append({
                "feature": c,
                "min": float(s.min()),
                "p01": float(q01),
                "median": float(q50),
                "p99": float(q99),
                "max": float(s.max()),
                "iqr": float(iqr),
                "missing_%": round(df[c].isna().mean() * 100, 2),
            })
        return pd.DataFrame(rows).sort_values("p99", ascending=False)

    print("Numeric columns considered:", len(num_cols))
    print("Columns targeted for outlier handling (focus_cols):", focus_cols)
    display(outlier_summary(df_b3, focus_cols))

Numeric columns considered: 6
Columns targeted for outlier handling (focus_cols): ['total_spend', 'avg_order_value', 'max_order_value', 'recency_days', 'n_orders']


,feature,min,p01,median,p99,max,iqr,missing_%
0,total_spend,1.5183,5.514,612.1369,132424.064034,989184.082000,3265.3331,0.0
2,max_order_value,1.5183,5.514,612.1369,40003.724088,187487.825000,2548.1410,0.0
1,avg_order_value,1.5183,5.514,612.1369,26111.291061,151704.902175,2113.8056,0.0
3,recency_days,0.0000,6.000,163.0000,900.000000,1126.000000,176.0000,0.0
4,n_orders,1.0000,1.000,1.0000,8.000000,28.000000,1.0000,0.0


In [56]:
# B.3 Coding Cell 2: Cap/winsorise outliers (1st–99th percentile by default)
# Output: cleaned_df_b3

if "cleaned_df" not in globals():
    print("Run B.1 and B.2 first to create cleaned_df.")
else:
    df_b3 = cleaned_df.copy()

    target_cols = [c for c in df_b3.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None

    protected = ["customer_id"] + ([target_col] if target_col else [])

    num_cols = df_b3.select_dtypes(include="number").columns.tolist()
    num_cols = [c for c in num_cols if c not in protected]

    # choose columns to cap (focus on monetary/behavioural)
    cap_cols = [c for c in num_cols if any(k in c.lower() for k in ["spend", "value", "amount", "total", "due", "subtotal", "price"])]
    for c in ["recency_days", "n_orders", "total_spend", "avg_order_value", "max_order_value"]:
        if c in num_cols and c not in cap_cols:
            cap_cols.append(c)

    # If nothing matched, cap all numeric (safe fallback)
    if len(cap_cols) == 0:
        cap_cols = num_cols

    LOW_Q, HIGH_Q = 0.01, 0.99

    caps = {}
    for c in cap_cols:
        lo = df_b3[c].quantile(LOW_Q)
        hi = df_b3[c].quantile(HIGH_Q)
        caps[c] = (float(lo), float(hi))
        df_b3[c] = df_b3[c].clip(lower=lo, upper=hi)

    cleaned_df_b3 = df_b3
    print(f"Winsorised columns at {int(LOW_Q*100)}th–{int(HIGH_Q*100)}th percentiles.")
    print("Capping thresholds (first 10 shown):", list(caps.items())[:10])
    display(cleaned_df_b3.head())

Winsorised columns at 1th–99th percentiles.
Capping thresholds (first 10 shown): [('total_spend', (5.514, 132424.06403399998)), ('avg_order_value', (5.514, 26111.29106147616)), ('max_order_value', (5.514, 40003.72408799979)), ('recency_days', (6.0, 900.0)), ('n_orders', (1.0, 8.0))]


,customer_id,recency_days,n_orders,total_spend,avg_order_value,max_order_value,territory_id,y_reorder_within_90d,territory_id__was_missing
0,00027a37-6f01-4a8f-bd81-23f2a6e7f525,32,2,6673.4266,3336.7133,3953.9884,unknown,0,1
1,0002bd5d-7aa8-403e-aa8c-d7134d92da55,32,1,937.5594,937.5594,937.5594,unknown,0,1
2,000421bb-5918-41c8-b95f-3bc0887876dd,296,1,35.6694,35.6694,35.6694,unknown,0,1
3,000a3e10-bb31-43db-adf5-a4450ea0cc7e,6,1,44.1779,44.1779,44.1779,unknown,0,1
4,000bc388-bef0-46fd-8f64-cd76f2350ef1,163,1,8.0444,8.0444,8.0444,unknown,0,1


In [57]:
# B.3 Coding Cell 3: Audit after capping (prove it worked)

import numpy as np

if "cleaned_df_b3" not in globals():
    print("Run B.3 Coding Cell 2 first to create cleaned_df_b3.")
else:
    df_after = cleaned_df_b3.copy()

    target_cols = [c for c in df_after.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None

    protected = ["customer_id"] + ([target_col] if target_col else [])
    num_cols = df_after.select_dtypes(include="number").columns.tolist()
    num_cols = [c for c in num_cols if c not in protected]

    # summarise numeric again
    rows = []
    for c in num_cols:
        s = df_after[c].dropna()
        if len(s) == 0:
            continue
        rows.append({
            "feature": c,
            "min": float(s.min()),
            "p01": float(s.quantile(0.01)),
            "median": float(s.quantile(0.5)),
            "p99": float(s.quantile(0.99)),
            "max": float(s.max()),
        })

    after_summary = pd.DataFrame(rows).sort_values("p99", ascending=False)
    print("Numeric summary after capping (top 25):")
    display(after_summary.head(25))

    # update pipeline variable for next sections
    cleaned_df = cleaned_df_b3
    print("cleaned_df updated. Shape:", cleaned_df.shape)

Numeric summary after capping (top 25):


,feature,min,p01,median,p99,max
2,total_spend,5.514,5.514,612.1369,132423.910055,132424.064034
4,max_order_value,5.514,5.514,612.1369,40002.410370,40003.724088
3,avg_order_value,5.514,5.514,612.1369,26111.112815,26111.291061
0,recency_days,6.000,6.000,163.0000,900.000000,900.000000
1,n_orders,1.000,1.000,1.0000,8.000000,8.000000
5,territory_id__was_missing,0.000,0.000,0.0000,1.000000,1.000000


cleaned_df updated. Shape: (23000, 9)


In [58]:
# B.3 Extra Cell 4: Before vs after comparison for a chosen numeric feature

if "cleaned_df_b3" not in globals():
    print("Run B.3 Coding Cell 2 first.")
else:
    # pick a feature to inspect (change if you want)
    candidate = None
    for c in ["total_spend", "avg_order_value", "max_order_value", "recency_days"]:
        if c in cleaned_df_b3.columns:
            candidate = c
            break

    if candidate is None:
        print("No common monetary/recency column found to compare.")
    else:
        before = cleaned_df[candidate].describe()
        after = cleaned_df_b3[candidate].describe()

        comp = pd.DataFrame({"before": before, "after": after})
        print("Feature compared:", candidate)
        display(comp)

Feature compared: total_spend


,before,after
count,23000.000000,23000.000000
mean,3731.993859,3731.993859
std,14784.521061,14784.521061
min,5.514000,5.514000
25%,60.752900,60.752900
50%,612.136900,612.136900
75%,3326.086000,3326.086000
max,132424.064034,132424.064034


In [59]:
data_cleaning_3_explanations = """
B.3 Fixing issue: Extreme outliers in numeric/monetary features (winsorisation / capping)

What the issue is
- Customer-level behavioural and monetary features (e.g., total_spend, avg_order_value, max_order_value, and sometimes recency_days or n_orders) are typically right-skewed.
- A small number of customers can have unusually large purchases or unusually high totals. These extreme values are outliers relative to the majority of customers.
- Outliers can originate from:
  - genuinely high-value customers (valid but rare),
  - data entry/ETL issues,
  - aggregation effects (multiple orders summed into total_spend),
  - or unusual business events (bulk purchases, refunds/adjustments depending on the source).

Why it is important to fix
- Outliers can dominate summary statistics (mean/variance) and distort the scale of features.
- Many models and preprocessing steps are sensitive to extreme values:
  - distance-based or margin-based models (e.g., kNN, SVM) are affected by feature scale,
  - linear/logistic models can be influenced by extreme points,
  - even tree-based models can create splits driven by a tiny number of extreme observations.
- If we do not handle outliers, the model may learn patterns that mainly fit rare extremes rather than typical customer behaviour, reducing generalisation on validation/testing data.

How the issue was fixed
- We applied winsorisation (quantile-based capping) to numeric features that are most likely to contain extreme values (monetary/behavioural columns).
- Specifically, values below the 1st percentile were capped to the 1st percentile value, and values above the 99th percentile were capped to the 99th percentile value.
- This approach preserves all rows (no deletions) while reducing the influence of extreme tails.

Impacts and benefits
- Improves numerical stability and makes the scale of features more representative of the majority of customers.
- Reduces the risk that the model overfits to a small number of extreme customers.
- Produces a more robust dataset for downstream transformations (e.g., scaling) and model training.

Limitations / considerations
- Capping reduces information about the exact magnitude of extreme customers, so a very small amount of signal may be compressed.
- Quantile thresholds (1% and 99%) are heuristic; for some features/datasets different thresholds may be appropriate.
- In a production setting, the capping thresholds should be calculated only on the training data and applied consistently to validation/testing to avoid information leakage.

Conclusion
- Handling outliers via winsorisation creates a cleaner, more robust numeric feature space, improving downstream model performance and reliability while keeping the dataset intact.
"""

In [60]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_3_explanations', value=data_cleaning_3_explanations)

### B.n Fixing `Remove duplicate rows and enforce correct data types`

> You can add more cells related to other issues in this section

In [61]:
# B.n Fixing (extra): Remove duplicate rows and enforce correct data types
# This is a general “catch-all” cleaning step to ensure the final cleaned_df is consistent.

if "cleaned_df" not in globals():
    print("Run B.1–B.3 first to create cleaned_df.")
else:
    df_bn = cleaned_df.copy()

    # detect target
    target_cols = [c for c in df_bn.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None

    # 1) Drop duplicate customer_id rows (should be 0, but enforce)
    if "customer_id" in df_bn.columns:
        before = len(df_bn)
        df_bn = df_bn.drop_duplicates(subset=["customer_id"], keep="first")
        after = len(df_bn)
        print(f"Dropped {before - after} duplicate customer_id rows.")
    else:
        # fallback: drop full duplicates
        before = len(df_bn)
        df_bn = df_bn.drop_duplicates()
        after = len(df_bn)
        print(f"Dropped {before - after} fully duplicated rows (no customer_id column found).")

    # 2) Enforce target type (binary int)
    if target_col is not None:
        df_bn[target_col] = pd.to_numeric(df_bn[target_col], errors="coerce").fillna(0).astype(int)
        # force into {0,1}
        df_bn[target_col] = (df_bn[target_col] > 0).astype(int)

    # 3) Convert numeric-looking object columns to numeric (safety)
    obj_cols = df_bn.select_dtypes(include=["object", "string"]).columns.tolist()
    obj_cols = [c for c in obj_cols if c not in ["customer_id", target_col]]

    converted = []
    for c in obj_cols:
        # attempt numeric conversion; only keep if conversion doesn't create many NaNs
        as_num = pd.to_numeric(df_bn[c], errors="coerce")
        # if at least 90% of non-null values can be numeric, convert it
        non_null = df_bn[c].notna().sum()
        ok = (as_num.notna().sum() / non_null) >= 0.90 if non_null > 0 else False
        if ok:
            df_bn[c] = as_num
            converted.append(c)

    print("Converted object -> numeric columns:", converted)

    # 4) Final null check
    total_null = int(df_bn.isna().sum().sum())
    print("Total remaining null cells:", total_null)

    cleaned_df = df_bn  # update pipeline variable
    print("cleaned_df updated. Shape:", cleaned_df.shape)
    display(cleaned_df.head())

Dropped 3881 duplicate customer_id rows.
Converted object -> numeric columns: []
Total remaining null cells: 0
cleaned_df updated. Shape: (19119, 9)


,customer_id,recency_days,n_orders,total_spend,avg_order_value,max_order_value,territory_id,y_reorder_within_90d,territory_id__was_missing
0,00027a37-6f01-4a8f-bd81-23f2a6e7f525,32,2,6673.4266,3336.7133,3953.9884,unknown,0,1
1,0002bd5d-7aa8-403e-aa8c-d7134d92da55,32,1,937.5594,937.5594,937.5594,unknown,0,1
2,000421bb-5918-41c8-b95f-3bc0887876dd,296,1,35.6694,35.6694,35.6694,unknown,0,1
3,000a3e10-bb31-43db-adf5-a4450ea0cc7e,6,1,44.1779,44.1779,44.1779,unknown,0,1
4,000bc388-bef0-46fd-8f64-cd76f2350ef1,163,1,8.0444,8.0444,8.0444,unknown,0,1


In [62]:
# B.n Extra Cell 1: Validate numeric ranges (impossible values)

if "cleaned_df" not in globals():
    print("Run B.1–B.n first to create cleaned_df.")
else:
    df = cleaned_df.copy()

    target_cols = [c for c in df.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None

    num_cols = df.select_dtypes(include="number").columns.tolist()
    num_cols = [c for c in num_cols if c not in ([target_col] if target_col else [])]

    issues = []

    # rule examples (adjust if you have different features)
    for c in num_cols:
        s = df[c]
        if "recency" in c.lower():
            neg = int((s < 0).sum())
            if neg > 0:
                issues.append((c, "recency should be >= 0", neg))
        if any(k in c.lower() for k in ["spend", "value", "amount", "total", "due", "price"]):
            neg = int((s < 0).sum())
            if neg > 0:
                issues.append((c, "monetary values should be >= 0", neg))
        if "n_orders" in c.lower():
            neg = int((s < 0).sum())
            if neg > 0:
                issues.append((c, "order counts should be >= 0", neg))

    if len(issues) == 0:
        print("No obvious impossible numeric values found (good).")
    else:
        issues_df = pd.DataFrame(issues, columns=["feature", "rule", "violations"])
        print("Potential numeric integrity issues:")
        display(issues_df)

No obvious impossible numeric values found (good).


In [63]:
# B.n Extra Cell 2: Fix invalid numeric values (clip to valid minimums)

if "cleaned_df" not in globals():
    print("Run previous steps first.")
else:
    df = cleaned_df.copy()

    for c in df.select_dtypes(include="number").columns:
        if "recency" in c.lower():
            df[c] = df[c].clip(lower=0)
        if any(k in c.lower() for k in ["spend", "value", "amount", "total", "due", "price"]):
            df[c] = df[c].clip(lower=0)
        if "n_orders" in c.lower():
            df[c] = df[c].clip(lower=0)

    cleaned_df = df
    print("Applied non-negative clipping rules where appropriate.")
    display(cleaned_df.head())

Applied non-negative clipping rules where appropriate.


,customer_id,recency_days,n_orders,total_spend,avg_order_value,max_order_value,territory_id,y_reorder_within_90d,territory_id__was_missing
0,00027a37-6f01-4a8f-bd81-23f2a6e7f525,32,2,6673.4266,3336.7133,3953.9884,unknown,0,1
1,0002bd5d-7aa8-403e-aa8c-d7134d92da55,32,1,937.5594,937.5594,937.5594,unknown,0,1
2,000421bb-5918-41c8-b95f-3bc0887876dd,296,1,35.6694,35.6694,35.6694,unknown,0,1
3,000a3e10-bb31-43db-adf5-a4450ea0cc7e,6,1,44.1779,44.1779,44.1779,unknown,0,1
4,000bc388-bef0-46fd-8f64-cd76f2350ef1,163,1,8.0444,8.0444,8.0444,unknown,0,1


In [64]:
# B.n Extra Cell 3: Final model-readiness checklist

if "cleaned_df" not in globals():
    print("Run B.1–B.n first.")
else:
    df = cleaned_df.copy()

    target_cols = [c for c in df.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None

    checklist = {}

    checklist["has_customer_id"] = "customer_id" in df.columns
    checklist["unique_customer_id"] = (df["customer_id"].is_unique if "customer_id" in df.columns else None)

    checklist["has_target"] = target_col is not None
    if target_col:
        checklist["target_is_binary_0_1"] = set(df[target_col].dropna().unique()).issubset({0, 1})
        checklist["target_prevalence"] = round(df[target_col].mean(), 4)

    checklist["total_null_cells"] = int(df.isna().sum().sum())
    checklist["n_rows"] = int(df.shape[0])
    checklist["n_cols"] = int(df.shape[1])

    # any object columns remaining (they can be one-hot encoded later; not necessarily bad)
    checklist["n_categorical_cols"] = int(len(df.select_dtypes(include=["object", "string", "category"]).columns))

    display(pd.DataFrame([checklist]))

,has_customer_id,unique_customer_id,has_target,target_is_binary_0_1,target_prevalence,total_null_cells,n_rows,n_cols,n_categorical_cols
0,True,True,True,True,0.0983,0,19119,9,2


In [65]:
# B.n Extra Cell 4: Separate predictors and target (avoid leakage)

if "cleaned_df" not in globals():
    print("Run B.1–B.n first.")
else:
    df = cleaned_df.copy()
    target_cols = [c for c in df.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None

    if target_col is None:
        print("No y_* target found.")
    else:
        y = df[target_col].copy()
        X = df.drop(columns=[target_col])

        print("X shape:", X.shape)
        print("y shape:", y.shape)
        print("y mean (prevalence):", round(y.mean(), 4))

        # store for next sections if you want
        X_clean = X
        y_clean = y

X shape: (19119, 8)
y shape: (19119,)
y mean (prevalence): 0.0983


In [66]:
data_cleaning_n_explanations = """
B.n Fixing issue: Final integrity + validity checks (duplicates, data types, invalid values, and pipeline readiness)

What the issue is
- After completing B.1–B.3 (missing values, categorical standardisation, and outlier capping), there are still common “last mile” risks that can break a modelling pipeline or bias evaluation:
  1) Duplicate rows / wrong dataset grain:
     - The modelling table is intended to be one row per customer. Any duplicate customer_id rows violate this assumption.
  2) Inconsistent data types:
     - Some columns may still be stored as text even though they represent numbers (e.g., "123.4" as a string).
  3) Invalid or impossible numeric values:
     - Some engineered fields can be logically constrained (e.g., recency_days should be >= 0, monetary values should not be negative, order counts should not be negative).
  4) Pipeline readiness problems:
     - The dataset should have a clean binary target, and the predictors should be separable from the target to avoid leakage in later steps.

Why it is important to fix
- Prevents training/evaluation errors:
  - Many transformations (scaling, correlation, model fitting) require correct numeric dtypes and will fail or behave incorrectly if numbers are stored as strings.
- Prevents biased splits and misleading metrics:
  - Duplicate customers can appear in multiple splits, creating overly optimistic validation/testing performance (data leakage through duplicates).
- Improves robustness for real-world usage:
  - In practical pipelines, it is better to enforce basic validity constraints (e.g., non-negative spend/recency) so the model behaves consistently on new data.

How it is fixed (what was done)
- Duplicate handling:
  - Dropped duplicate rows using customer_id to enforce one row per customer (correct modelling grain).
- Type enforcement:
  - Converted numeric-looking text columns to numeric dtype when conversion was reliable (most values parseable).
- Target enforcement:
  - Ensured the target y_* is numeric, binary, and restricted to {0, 1}.
- Validity checks / corrections:
  - Applied simple non-negativity rules (clipping) for logically constrained features such as recency_days, monetary amounts, and order-count style fields.
- Readiness checks:
  - Performed a final “model-readiness checklist” (null count, unique customer_id, binary target, and shapes), and optionally separated X and y to prevent accidental leakage.

Impact
- Guarantees correct dataset grain and reduces the risk of split contamination.
- Ensures consistent datatypes and valid numeric ranges, improving stability of downstream encoding/scaling and model training.
- Produces a clean, reproducible, model-ready dataset that can be safely used in Section C (splitting) and Section E (data preparation for modelling).
"""

In [67]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_n_explanations', value=data_cleaning_n_explanations)

---
## C. Split Datasets


In [68]:
'''36106
# <Student to fill this section>
training_df = None
validation_df = None
testing_df = None
'''

'36106\n# <Student to fill this section>\ntraining_df = None\nvalidation_df = None\ntesting_df = None\n'

In [69]:
# <Student to fill this section>
# Section C: Split datasets into training / validation / testing
# Strategy:
# - Prefer time-based split using last_order (prevents leakage across time).
# - Fallback to random split if last_order is not available.

import numpy as np

if "cleaned_df" not in globals():
    print("Run Section B first to create cleaned_df.")
else:
    df = cleaned_df.copy()

    # detect target column
    target_cols = [c for c in df.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None
    if target_col is None:
        raise ValueError("No target column found (expected a y_* column). Create label in A.1 first.")

    # --- SETTINGS ---
    TRAIN_FRAC = 0.70
    VAL_FRAC = 0.15
    TEST_FRAC = 0.15
    SEED = 42

    assert abs((TRAIN_FRAC + VAL_FRAC + TEST_FRAC) - 1.0) < 1e-9, "Fractions must sum to 1."

    # --- Prefer time-based split if we have a usable timestamp feature ---
    time_col = None
    if "last_order" in df.columns:
        time_col = "last_order"

    if time_col is not None:
        df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
        df = df.dropna(subset=[time_col]).sort_values(time_col).reset_index(drop=True)

        n = len(df)
        train_end = int(n * TRAIN_FRAC)
        val_end = int(n * (TRAIN_FRAC + VAL_FRAC))

        training_df = df.iloc[:train_end].copy()
        validation_df = df.iloc[train_end:val_end].copy()
        testing_df = df.iloc[val_end:].copy()

        split_method = f"time-based using {time_col}"
    else:
        # --- Random split fallback ---
        df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

        n = len(df)
        train_end = int(n * TRAIN_FRAC)
        val_end = int(n * (TRAIN_FRAC + VAL_FRAC))

        training_df = df.iloc[:train_end].copy()
        validation_df = df.iloc[train_end:val_end].copy()
        testing_df = df.iloc[val_end:].copy()

        split_method = f"random shuffle (seed={SEED})"

    # --- Basic checks ---
    print("Split method:", split_method)
    print("Shapes:")
    print("  training_df:", training_df.shape)
    print("  validation_df:", validation_df.shape)
    print("  testing_df:", testing_df.shape)

    # Check no overlap by customer_id
    if "customer_id" in df.columns:
        tr = set(training_df["customer_id"])
        va = set(validation_df["customer_id"])
        te = set(testing_df["customer_id"])
        print("Customer overlap checks (should be 0):")
        print("  train ∩ val:", len(tr & va))
        print("  train ∩ test:", len(tr & te))
        print("  val ∩ test:", len(va & te))

    # Check target balance
    def prevalence(x):
        return round(x[target_col].mean(), 4)

    print("Target prevalence:")
    print("  train:", prevalence(training_df))
    print("  val:  ", prevalence(validation_df))
    print("  test: ", prevalence(testing_df))

Split method: random shuffle (seed=42)
Shapes:
  training_df: (13383, 9)
  validation_df: (2868, 9)
  testing_df: (2868, 9)
Customer overlap checks (should be 0):
  train ∩ val: 0
  train ∩ test: 0
  val ∩ test: 0
Target prevalence:
  train: 0.0974
  val:   0.0941
  test:  0.1067


In [70]:
# (Optional) Inspect time boundaries if time-based split was used

if "training_df" not in globals():
    print("Run the split cell first.")
else:
    if "last_order" in training_df.columns:
        print("Train last_order range:", training_df["last_order"].min(), "->", training_df["last_order"].max())
        print("Val last_order range:  ", validation_df["last_order"].min(), "->", validation_df["last_order"].max())
        print("Test last_order range: ", testing_df["last_order"].min(), "->", testing_df["last_order"].max())
    else:
        print("No last_order column available; random split was used.")

No last_order column available; random split was used.


In [71]:
# C Extra Cell 2: Leakage sanity check on split datasets

if "training_df" not in globals():
    print("Run the split cell first.")
else:
    # detect target
    target_cols = [c for c in training_df.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None

    if target_col is None:
        print("No y_* target column found in training_df.")
    else:
        # columns that look like leakage risks
        suspicious_keywords = ["next", "future", "label", "target", "reorder_within", "churn", "y_"]
        suspicious = [c for c in training_df.columns if any(k in c.lower() for k in suspicious_keywords) and c != target_col]

        print("Target column:", target_col)
        print("Suspicious non-target columns (should be empty):", suspicious)

        # confirm target is not in features_list if you have it
        if "features_list" in globals():
            in_features = [c for c in features_list if c == target_col or c.startswith("y_")]
            print("Target accidentally in features_list (should be empty):", in_features)

Target column: y_reorder_within_90d
Suspicious non-target columns (should be empty): ['recency_days', 'territory_id', 'territory_id__was_missing']
Target accidentally in features_list (should be empty): []


In [72]:
# C Extra Cell 3: Drift proxy (numeric feature means) across splits

import numpy as np

if "training_df" not in globals():
    print("Run the split cell first.")
else:
    # detect target
    target_cols = [c for c in training_df.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None

    # choose numeric predictors only
    def numeric_predictors(df):
        cols = df.select_dtypes(include="number").columns.tolist()
        cols = [c for c in cols if c not in (["customer_id"] + ([target_col] if target_col else []))]
        return cols

    num_cols = numeric_predictors(training_df)
    if len(num_cols) == 0:
        print("No numeric predictors found for drift check.")
    else:
        comp = pd.DataFrame({
            "feature": num_cols,
            "train_mean": training_df[num_cols].mean().values,
            "val_mean": validation_df[num_cols].mean().values,
            "test_mean": testing_df[num_cols].mean().values,
        })
        comp["test_minus_train"] = comp["test_mean"] - comp["train_mean"]
        comp["pct_diff_test_vs_train"] = (comp["test_minus_train"] / comp["train_mean"].replace(0, np.nan) * 100).round(2)

        display(comp.sort_values("pct_diff_test_vs_train", ascending=False).head(20))
        print("Note: large differences suggest temporal drift; time-based split is appropriate.")

,feature,train_mean,val_mean,test_mean,test_minus_train,pct_diff_test_vs_train
2,total_spend,3779.055357,3981.031375,4406.289652,627.234295,16.60
4,max_order_value,1771.043853,1810.485027,1979.953147,208.909294,11.80
3,avg_order_value,1363.263719,1398.741777,1497.224169,133.960450,9.83
1,n_orders,1.597773,1.586820,1.665969,0.068196,4.27
0,recency_days,187.693566,194.134937,188.179916,0.486350,0.26
5,territory_id__was_missing,0.518494,0.519874,0.508020,-0.010474,-2.02


Note: large differences suggest temporal drift; time-based split is appropriate.


In [73]:
# C Extra Cell 4: Target prevalence summary table

if "training_df" not in globals():
    print("Run the split cell first.")
else:
    target_cols = [c for c in training_df.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None

    if target_col is None:
        print("No y_* target column found.")
    else:
        prev = pd.DataFrame({
            "split": ["train", "val", "test"],
            "rows": [len(training_df), len(validation_df), len(testing_df)],
            "target_mean": [
                training_df[target_col].mean(),
                validation_df[target_col].mean(),
                testing_df[target_col].mean(),
            ],
        })
        prev["target_mean"] = prev["target_mean"].round(4)
        display(prev)

,split,rows,target_mean
0,train,13383,0.0974
1,val,2868,0.0941
2,test,2868,0.1067


In [74]:
import altair as alt
from IPython.display import display

# ----------------------------
# Safety checks
# ----------------------------
if "territory_id" not in training_df.columns:
    print("territory_id not found")
elif target_col not in training_df.columns:
    print(f"{target_col} not found in training_df")
else:

    seg = (
        training_df
        .groupby("territory_id")[target_col]
        .agg(["mean", "count"])
        .reset_index()
        .rename(columns={"mean": "target_rate", "count": "n"})
    )

    seg["territory_id"] = seg["territory_id"].fillna("<<MISSING>>").astype(str)

    chart = (
        alt.Chart(seg)
        .mark_bar()
        .encode(
            x=alt.X("territory_id:N", title="Territory"),
            y=alt.Y(
                "target_rate:Q",
                title="Reorder rate (class=1)",
                axis=alt.Axis(format=".0%")
            ),
            color=alt.Color(
                "target_rate:Q",
                scale=alt.Scale(scheme="blues")
            ),
            tooltip=["territory_id:N", "target_rate:Q", "n:Q"]
        )
        .properties(
            title=f"Target Rate by Territory ({target_col})",
            width=500,
            height=300
        )
    )

    display(chart)

alt.Chart(...)

In [75]:
data_splitting_explanations = """
Best splitting strategy for this dataset (updated)

This project is a customer-level classification task derived from historical transaction data. Because purchasing behaviour changes over time (seasonality, promotions, customer lifecycle), the most reliable way to evaluate performance is to mimic a real deployment setting: train on the past and test on the future.

Primary strategy: time-aware split (preferred)
- If a usable time marker is available (e.g., each customer’s last_order date), customers are sorted chronologically and split into:
  - Training set (earliest period): used to fit the model using historical behaviour.
  - Validation set (middle period): used to tune choices such as feature engineering and preprocessing decisions.
  - Testing set (latest period): used once for final, unbiased evaluation and best approximates future performance.
Why this is best:
- It reduces temporal leakage, where information patterns from a later period could indirectly influence earlier training.
- It provides a more realistic estimate of how the model will perform when predicting on new/future customer behaviour.

Fallback strategy: reproducible random split
- If a reliable time column is not available, a random split is used with a fixed seed for reproducibility.
- Even in the random split case, the key rule is enforced: each customer_id must appear in only one split to prevent the same customer’s information leaking across training/validation/testing.

Additional checks performed
- Verified there is no customer_id overlap between splits (train ∩ val, train ∩ test, val ∩ test should be 0).
- Compared target prevalence across splits to confirm that the class balance is reasonably consistent and evaluation remains meaningful.

Conclusion
- A time-based split is the most appropriate strategy for transaction-derived datasets, and the random split is a controlled backup when time ordering is not available.


"""

In [76]:
# Do not modify this code
print_tile(size="h3", key='data_splitting_explanations', value=data_splitting_explanations)

---
## D. Feature Engineering

In [77]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  training_df_eng = training_df.copy()
  validation_df_eng = validation_df.copy()
  testing_df_eng = testing_df.copy()
except Exception as e:
  print(e)

### D.1 New Feature "`log_total_spend`"



In [78]:
# D.1 New Feature: log_total_spend (reduce spend skew)
# Adds: log_total_spend to training_df_eng / validation_df_eng / testing_df_eng

import numpy as np
import pandas as pd

if "training_df_eng" not in globals():
    print("Run Section C and the 'Create copy of datasets' cell first (training_df_eng not found).")
else:
    required = ["total_spend"]
    for name, df in [("training_df_eng", training_df_eng),
                     ("validation_df_eng", validation_df_eng),
                     ("testing_df_eng", testing_df_eng)]:
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise KeyError(f"{name} missing columns: {missing}")

        # clip just in case any negative values exist
        df["log_total_spend"] = np.log1p(df["total_spend"].clip(lower=0))

    print("Created feature: log_total_spend")
    display(training_df_eng[["total_spend", "log_total_spend"]].head())

    # Extra audit
    print("Train missing %:", round(training_df_eng["log_total_spend"].isna().mean() * 100, 4))
    print("Train min/median/max:",
          float(training_df_eng["log_total_spend"].min()),
          float(training_df_eng["log_total_spend"].median()),
          float(training_df_eng["log_total_spend"].max()))

Created feature: log_total_spend


,total_spend,log_total_spend
0,612.1369,6.418588
1,6401.3203,8.764416
2,71.7919,4.287605
3,3352.6455,8.117803
4,62.9519,4.158131


Train missing %: 0.0
Train min/median/max: 1.8739537068540417 6.409554459280651 11.793772209961789


In [79]:
# D.1 EXTRA: diagnostics + simple plot + target relationship check (train only)

if "log_total_spend" not in training_df_eng.columns:
    print("Run D.1 feature creation first.")
else:
    col = "log_total_spend"
    target_col = [c for c in training_df_eng.columns if c.startswith("y_")][0]

    # 1) Distribution summary
    print("D.1 EXTRA — Distribution summary (train):")
    display(training_df_eng[col].describe().to_frame("train"))

    # 2) Check how much skew was reduced (compare raw vs log)
    print("\nSkewness comparison (train):")
    raw_skew = training_df_eng["total_spend"].skew()
    log_skew = training_df_eng[col].skew()
    display(pd.DataFrame({"feature": ["total_spend", "log_total_spend"],
                          "skew": [raw_skew, log_skew]}))

    # 3) Quick target rate by log-spend quantile bins (evidence for usefulness)
    tmp = training_df_eng[[col, target_col]].dropna().copy()
    tmp["bin"] = pd.qcut(tmp[col], q=10, duplicates="drop")
    by_bin = tmp.groupby("bin").agg(
        mean_log_spend=(col, "mean"),
        target_rate=(target_col, "mean"),
        count=(target_col, "size")
    ).reset_index()
    print("\nTarget rate by log_total_spend bins (train):")
    display(by_bin)

    # Optional chart if Altair is available
    try:
        import altair as alt
        alt.data_transformers.disable_max_rows()
        chart = alt.Chart(by_bin).mark_line(point=True).encode(
            x=alt.X("mean_log_spend:Q", title="Mean log_total_spend (bin)"),
            y=alt.Y("target_rate:Q", title=f"Mean {target_col} (target rate)"),
            tooltip=["count:Q", "mean_log_spend:Q", "target_rate:Q"]
        ).properties(title="Target rate vs log_total_spend (train)", width=700, height=300)
        chart
    except Exception as e:
        print("Altair plot skipped:", e)

D.1 EXTRA — Distribution summary (train):


,train
count,13383.000000
mean,6.020634
std,2.266989
min,1.873954
25%,4.123141
50%,6.409554
75%,8.021725
max,11.793772



Skewness comparison (train):


,feature,skew
0,total_spend,7.636988
1,log_total_spend,0.065983



Target rate by log_total_spend bins (train):


,bin,mean_log_spend,target_rate,count
0,"(1.8730000000000002, 3.354]",2.535079,0.002876,1391
1,"(3.354, 3.811]",3.664732,0.012829,1481
2,"(3.811, 4.347]",4.144553,0.081356,1180
3,"(4.347, 4.704]",4.485727,0.118280,1302
4,"(4.704, 6.41]",5.256668,0.315257,1383
5,"(6.41, 7.19]",6.737629,0.014683,1294
6,"(7.19, 7.877]",7.602656,0.010409,1345
7,"(7.877, 8.283]",8.056392,0.012561,1433
8,"(8.283, 8.767]",8.526959,0.101215,1235
9,"(8.767, 11.794]",9.409827,0.312173,1339


In [80]:
feature_engineering_1_explanations = """
D.1 New Feature: log_total_spend

What the feature is
- log_total_spend is created as log1p(total_spend), i.e., log(1 + total_spend).
- This converts the raw monetary spend variable into a log scale while safely handling zero values.

Why it is important
- Spend-related variables are usually heavily right-skewed: most customers spend small/moderate amounts, while a small number spend extremely large amounts.
- If we use total_spend directly, extreme values can dominate the scale of the feature and overly influence models (especially linear/logistic models and distance-based methods).
- The log transform compresses large values and spreads out small values, making the distribution more balanced and easier for models to learn from.

Impacts on modelling
- Improves numerical stability and makes scaling (e.g., StandardScaler) more meaningful because the feature distribution becomes less skewed.
- Reduces the influence of outliers even after winsorisation, helping the model generalise better to typical customers rather than being driven by a few high-spend cases.
- Can improve performance for models that assume roughly linear relationships between predictors and the log-odds of the target (e.g., logistic regression).

Leakage considerations
- log_total_spend is derived only from historical order totals already available in the dataset (no future information is used), so it does not introduce leakage.

Conclusion
- log_total_spend is a simple, interpretable transformation that retains the business meaning of customer value while making the feature more model-friendly and robust.
"""

In [81]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_1_explanations', value=feature_engineering_1_explanations)

### D.2 New Feature "`spend_per_order = total_spend / n_orders`"



In [82]:
# D.2 New Feature: spend_per_order = total_spend / n_orders
# Adds: spend_per_order to training_df_eng / validation_df_eng / testing_df_eng

import numpy as np
import pandas as pd

if "training_df_eng" not in globals():
    print("Run Section C and the 'Create copy of datasets' cell first (training_df_eng not found).")
else:
    required = ["total_spend", "n_orders"]
    for name, df in [("training_df_eng", training_df_eng),
                     ("validation_df_eng", validation_df_eng),
                     ("testing_df_eng", testing_df_eng)]:
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise KeyError(f"{name} missing columns: {missing}")

        denom = df["n_orders"].replace(0, np.nan)  # avoid divide-by-zero
        df["spend_per_order"] = (df["total_spend"] / denom).fillna(0)

    print("Created feature: spend_per_order")
    display(training_df_eng[["total_spend", "n_orders", "spend_per_order"]].head())

    # Extra audit
    n_inf = int(np.isinf(training_df_eng["spend_per_order"].to_numpy()).sum())
    print("Train inf count (should be 0):", n_inf)
    print("Train min/median/max:",
          float(training_df_eng["spend_per_order"].min()),
          float(training_df_eng["spend_per_order"].median()),
          float(training_df_eng["spend_per_order"].max()))

Created feature: spend_per_order


,total_spend,n_orders,spend_per_order
0,612.1369,1,612.13690
1,6401.3203,2,3200.66015
2,71.7919,1,71.79190
3,3352.6455,2,1676.32275
4,62.9519,1,62.95190


Train inf count (should be 0): 0
Train min/median/max: 5.514 596.689 33106.016008499995


In [83]:
# D.2 EXTRA: edge-case checks + compare against avg_order_value + target-rate bins

if "spend_per_order" not in training_df_eng.columns:
    print("Run D.2 feature creation first.")
else:
    col = "spend_per_order"
    target_col = [c for c in training_df_eng.columns if c.startswith("y_")][0]

    # 1) Edge-case sanity checks
    print("D.2 EXTRA — Edge cases (train):")
    n_zero_orders = int((training_df_eng["n_orders"] == 0).sum()) if "n_orders" in training_df_eng.columns else None
    n_zero_spend_per_order = int((training_df_eng[col] == 0).sum())
    n_negative = int((training_df_eng[col] < 0).sum())
    print("n_orders == 0 rows:", n_zero_orders)
    print("spend_per_order == 0 rows:", n_zero_spend_per_order)
    print("spend_per_order < 0 rows (should be 0):", n_negative)

    # 2) Relationship with avg_order_value (they may be similar; verify)
    if "avg_order_value" in training_df_eng.columns:
        tmp = training_df_eng[[col, "avg_order_value"]].dropna()
        corr = tmp[col].corr(tmp["avg_order_value"])
        print("\nCorrelation with avg_order_value (train):", round(float(corr), 4))
        display(tmp.describe())

    # 3) Target rate by spend_per_order bins
    tmp = training_df_eng[[col, target_col]].dropna().copy()
    # Use quantiles; duplicates drop if many identical values
    tmp["bin"] = pd.qcut(tmp[col], q=10, duplicates="drop")
    by_bin = tmp.groupby("bin").agg(
        mean_spend_per_order=(col, "mean"),
        target_rate=(target_col, "mean"),
        count=(target_col, "size")
    ).reset_index()
    print("\nTarget rate by spend_per_order bins (train):")
    display(by_bin)

    # Optional chart
    try:
        import altair as alt
        alt.data_transformers.disable_max_rows()
        chart = alt.Chart(by_bin).mark_line(point=True).encode(
            x=alt.X("mean_spend_per_order:Q", title="Mean spend_per_order (bin)"),
            y=alt.Y("target_rate:Q", title=f"Mean {target_col} (target rate)"),
            tooltip=["count:Q", "mean_spend_per_order:Q", "target_rate:Q"]
        ).properties(title="Target rate vs spend_per_order (train)", width=700, height=300)
        chart
    except Exception as e:
        print("Altair plot skipped:", e)

D.2 EXTRA — Edge cases (train):
n_orders == 0 rows: 0
spend_per_order == 0 rows: 0
spend_per_order < 0 rows (should be 0): 0

Correlation with avg_order_value (train): 0.9569


,spend_per_order,avg_order_value
count,13383.000000,13383.000000
mean,1339.356365,1363.263719
std,2944.084773,3035.173606
min,5.514000,5.514000
25%,46.343750,46.100625
50%,596.689000,596.689000
75%,2104.187400,2103.588300
max,33106.016008,26111.291061



Target rate by spend_per_order bins (train):


,bin,mean_spend_per_order,target_rate,count
0,"(5.513, 26.277]",13.173670,0.049097,1385
1,"(26.277, 41.957]",34.404136,0.097692,1300
2,"(41.957, 62.952]",49.642448,0.174069,1396
3,"(62.952, 83.405]",73.908162,0.120690,1276
4,"(83.405, 596.689]",161.377944,0.110375,1359
5,"(596.689, 1071.005]",748.462006,0.015015,1332
6,"(1071.005, 1802.714]",1442.439899,0.030967,1324
7,"(1802.714, 2458.084]",2126.805928,0.107865,1335
8,"(2458.084, 2719.223]",2596.227741,0.077038,1337
9,"(2719.223, 33106.016]",6169.925919,0.188947,1339


In [84]:
feature_engineering_2_explanations = """
D.2 New Feature: spend_per_order

What the feature is
- spend_per_order is calculated as:
    spend_per_order = total_spend / n_orders
  (with a safe handling for divide-by-zero, e.g., n_orders=0 -> spend_per_order set to 0).
- It measures the average spend intensity per order at the customer level.

Why it is important
- total_spend alone can be high for two very different customer behaviours:
  1) many small orders (high frequency, lower basket size), or
  2) a few large orders (low frequency, high basket size).
- spend_per_order separates these behaviours and captures “basket size / value per transaction”, which is often predictive of repeat purchasing patterns.

Impacts on modelling
- Adds additional signal beyond the basic RFM set because it combines monetary and frequency behaviour into one interpretable metric.
- Can improve classification by helping the model distinguish customers who purchase often but cheaply versus customers who purchase rarely but spend a lot.
- Produces a continuous numeric feature that is easy to scale and use in most ML models.

Leakage considerations
- spend_per_order is derived only from aggregated historical behaviour (total_spend and n_orders), so it does not use future information and does not create leakage.

Conclusion
- spend_per_order is a useful engineered feature because it captures customer value per transaction, complements total_spend and n_orders, and remains simple and explainable.
"""

In [85]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_2_explanations', value=feature_engineering_2_explanations)

### D.3 New Feature "`orders_per_year_active`"



In [86]:
# D.3 New Feature: orders_per_year_active = n_orders / ( (recency_days+1) / 365 )
# Adds: orders_per_year_active to training_df_eng / validation_df_eng / testing_df_eng

import numpy as np
import pandas as pd

if "training_df_eng" not in globals():
    print("Run Section C and the 'Create copy of datasets' cell first (training_df_eng not found).")
else:
    required = ["n_orders", "recency_days"]
    for name, df in [("training_df_eng", training_df_eng),
                     ("validation_df_eng", validation_df_eng),
                     ("testing_df_eng", testing_df_eng)]:
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise KeyError(f"{name} missing columns: {missing}")

        # +1 avoids division by zero when recency_days=0; clip ensures no weird negatives
        active_days = (df["recency_days"].clip(lower=0) + 1).clip(lower=1)
        df["orders_per_year_active"] = df["n_orders"] / (active_days / 365.0)

    print("Created feature: orders_per_year_active")
    display(training_df_eng[["recency_days", "n_orders", "orders_per_year_active"]].head())

    # Extra audit
    n_inf = int(np.isinf(training_df_eng["orders_per_year_active"].to_numpy()).sum())
    print("Train inf count (should be 0):", n_inf)
    print("Train min/median/max:",
          float(training_df_eng["orders_per_year_active"].min()),
          float(training_df_eng["orders_per_year_active"].median()),
          float(training_df_eng["orders_per_year_active"].max()))

Created feature: orders_per_year_active


,recency_days,n_orders,orders_per_year_active
0,285,1,1.276224
1,202,2,3.596059
2,18,1,19.210526
3,56,2,12.807018
4,268,1,1.356877


Train inf count (should be 0): 0
Train min/median/max: 0.40510543840177576 3.1465517241379306 417.1428571428571


In [87]:
# D.3 EXTRA: stability checks + outlier inspection + target-rate bins

if "orders_per_year_active" not in training_df_eng.columns:
    print("Run D.3 feature creation first.")
else:
    col = "orders_per_year_active"
    target_col = [c for c in training_df_eng.columns if c.startswith("y_")][0]

    # 1) Basic distribution + outlier inspection
    print("D.3 EXTRA — Distribution summary (train):")
    display(training_df_eng[col].describe().to_frame("train"))

    # Show top 10 extreme values to confirm they make sense
    print("\nTop 10 orders_per_year_active rows (train):")
    display(training_df_eng.sort_values(col, ascending=False)[
        ["customer_id", "recency_days", "n_orders", col, target_col]
    ].head(10))

    # 2) Check if feature is dominated by very small recency_days (can inflate rate)
    # If recency_days is tiny, orders_per_year_active can become huge.
    tmp = training_df_eng[["recency_days", col]].dropna().copy()
    tmp["recency_bucket"] = pd.cut(tmp["recency_days"], bins=[-1, 7, 30, 90, 180, 365, 99999],
                                   labels=["<=7d", "8-30d", "31-90d", "91-180d", "181-365d", ">365d"])
    bucket_stats = tmp.groupby("recency_bucket").agg(
        mean_orders_per_year_active=(col, "mean"),
        median_orders_per_year_active=(col, "median"),
        count=(col, "size")
    ).reset_index()
    print("\norders_per_year_active by recency bucket (train):")
    display(bucket_stats)

    # 3) Target rate by orders_per_year_active bins
    tmp2 = training_df_eng[[col, target_col]].dropna().copy()
    tmp2["bin"] = pd.qcut(tmp2[col], q=10, duplicates="drop")
    by_bin = tmp2.groupby("bin").agg(
        mean_orders_per_year_active=(col, "mean"),
        target_rate=(target_col, "mean"),
        count=(target_col, "size")
    ).reset_index()
    print("\nTarget rate by orders_per_year_active bins (train):")
    display(by_bin)

    # Optional chart
    try:
        import altair as alt
        alt.data_transformers.disable_max_rows()
        chart = alt.Chart(by_bin).mark_line(point=True).encode(
            x=alt.X("mean_orders_per_year_active:Q", title="Mean orders_per_year_active (bin)"),
            y=alt.Y("target_rate:Q", title=f"Mean {target_col} (target rate)"),
            tooltip=["count:Q", "mean_orders_per_year_active:Q", "target_rate:Q"]
        ).properties(title="Target rate vs orders_per_year_active (train)", width=700, height=300)
        chart
    except Exception as e:
        print("Altair plot skipped:", e)

D.3 EXTRA — Distribution summary (train):


,train
count,13383.000000
mean,7.424935
std,17.754053
min,0.405105
25%,1.622222
50%,3.146552
75%,6.886792
max,417.142857



Top 10 orders_per_year_active rows (train):


,customer_id,recency_days,n_orders,orders_per_year_active,y_reorder_within_90d
10554,709e717a-95c2-4756-8d44-ad203a624308,6,8,417.142857,1
8244,974675c2-1a2c-4358-8fb0-ca5afe78682f,6,8,417.142857,1
12914,5acd6ee9-cc74-4534-9888-544b7e427157,6,8,417.142857,1
5853,c7140abd-ff7a-4d92-8f0e-f754c981d8d2,6,8,417.142857,1
1704,697de217-ae4e-438e-96bb-3945a8219b8d,6,8,417.142857,1
8931,f363cdd3-80f1-4776-bafd-d8d4905352ba,6,8,417.142857,1
10282,49e68b87-02e2-4405-8916-06adcbf94259,6,8,417.142857,1
10709,2ff48c13-c8d9-469f-ade3-01937fd26e17,6,8,417.142857,1
11435,7761221b-e002-4712-8e1e-c09cb9a0ac4c,6,8,417.142857,1
856,52c2af45-cdc7-40f1-8910-6d4d9eda8fea,8,8,324.444444,1



orders_per_year_active by recency bucket (train):


,recency_bucket,mean_orders_per_year_active,median_orders_per_year_active,count
0,<=7d,98.512755,52.142857,175
1,8-30d,35.765105,26.071429,469
2,31-90d,11.600514,9.012346,2978
3,91-180d,5.305989,3.762887,3681
4,181-365d,1.943795,1.580087,5381
5,>365d,1.168963,0.869048,699



Target rate by orders_per_year_active bins (train):


,bin,mean_orders_per_year_active,target_rate,count
0,"(0.404, 1.13]",0.912804,0.000000,1341
1,"(1.13, 1.437]",1.271562,0.002959,1352
2,"(1.437, 1.862]",1.638869,0.024169,1324
3,"(1.862, 2.401]",2.125687,0.031387,1370
4,"(2.401, 3.147]",2.775072,0.075988,1316
5,"(3.147, 4.148]",3.627828,0.096536,1357
6,"(4.148, 5.748]",4.872738,0.103738,1311
7,"(5.748, 8.488]",7.016252,0.118545,1375
8,"(8.488, 15.423]",11.252560,0.141104,1304
9,"(15.423, 417.143]",39.053886,0.382596,1333


In [88]:
feature_engineering_3_explanations = """
D.3 New Feature: orders_per_year_active

What the feature is
- orders_per_year_active is calculated as:
    orders_per_year_active = n_orders / ((recency_days + 1) / 365)
- This approximates an order rate (orders per year) while avoiding division by zero using (recency_days + 1).

Why it is important
- n_orders measures frequency but it does not account for time.
  Example: 3 orders over 30 days is very different from 3 orders over 900 days.
- By normalising purchase count by the customer’s “active time window” (proxied by recency_days), this feature captures purchasing velocity.
- Purchase velocity is often a strong indicator of future reordering: customers who buy more frequently per unit time are more likely to reorder again.

Impacts on modelling
- Adds a time-adjusted behavioural signal that complements recency_days and n_orders.
- Helps the model identify recently active, fast-purchasing customers versus customers whose purchases are spread out over long periods.
- Because the feature can become large when recency_days is very small, it may still be skewed; this is acceptable, but it should be monitored (and optionally log-transformed later if needed).

Leakage considerations
- This feature uses only historical aggregates already present in the modelling table (n_orders and recency_days).
- It does not use any post-event or “future” information, so it is leakage-safe.

Conclusion
- orders_per_year_active is a meaningful engineered feature because it encodes purchase frequency in a time-aware way, improving the model’s ability to distinguish high-velocity repeat buyers from low-velocity customers.
"""

In [89]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_3_explanations', value=feature_engineering_3_explanations)

### D.n Fixing "`Leakage guard + column consistency across splits`"

> You can add more cells related to new features in this section

In [90]:
# D.n Fix: Leakage guard + column consistency across splits
# - Ensures engineered features exist in all splits
# - Ensures customer_id is NOT used as a predictor
# - Ensures train/val/test have identical predictor columns

import numpy as np
import pandas as pd

if "training_df_eng" not in globals():
    print("Run Section C and the dataset copy cell first (training_df_eng not found).")
else:
    # 1) Detect target
    target_cols = [c for c in training_df_eng.columns if c.startswith("y_")]
    if len(target_cols) == 0:
        raise ValueError("No y_* target column found in training_df_eng.")
    target_col = target_cols[0]

    # 2) Columns that must never be used as predictors
    leak_prone_cols = ["customer_id", target_col]

    # 3) Ensure all engineered features exist in all splits
    engineered = ["log_total_spend", "spend_per_order", "orders_per_year_active"]
    for feat in engineered:
        for name, df in [("train", training_df_eng),
                         ("val", validation_df_eng),
                         ("test", testing_df_eng)]:
            if feat not in df.columns:
                raise KeyError(f"Engineered feature '{feat}' is missing from {name} split. "
                               f"Run D.1–D.3 feature creation on *_df_eng (not only one split).")

    # 4) Build predictor lists for each split (exclude ID + target)
    def predictor_cols(df):
        return [c for c in df.columns if c not in leak_prone_cols]

    cols_train = predictor_cols(training_df_eng)
    cols_val = predictor_cols(validation_df_eng)
    cols_test = predictor_cols(testing_df_eng)

    # 5) Check same columns across splits
    set_train, set_val, set_test = set(cols_train), set(cols_val), set(cols_test)

    missing_in_val = sorted(list(set_train - set_val))
    missing_in_test = sorted(list(set_train - set_test))
    extra_in_val = sorted(list(set_val - set_train))
    extra_in_test = sorted(list(set_test - set_train))

    print("D.n Consistency check:")
    print("Missing in val (should be []):", missing_in_val)
    print("Missing in test (should be []):", missing_in_test)
    print("Extra in val (should be []):", extra_in_val)
    print("Extra in test (should be []):", extra_in_test)

    if missing_in_val or missing_in_test or extra_in_val or extra_in_test:
        raise ValueError("Predictor columns are inconsistent across splits. Fix before modelling.")

    # 6) Enforce identical column order (critical for downstream modelling)
    # Use training column order as the master order
    validation_df_eng = validation_df_eng[cols_train + [target_col, "customer_id"] if "customer_id" in validation_df_eng.columns else cols_train + [target_col]]
    testing_df_eng = testing_df_eng[cols_train + [target_col, "customer_id"] if "customer_id" in testing_df_eng.columns else cols_train + [target_col]]

    print("\n✅ D.n passed: no leakage columns used as predictors and splits have consistent columns.")
    print("Predictor column count:", len(cols_train))

D.n Consistency check:
Missing in val (should be []): []
Missing in test (should be []): []
Extra in val (should be []): []
Extra in test (should be []): []

✅ D.n passed: no leakage columns used as predictors and splits have consistent columns.
Predictor column count: 10


In [91]:
# D.n Fix (Extra Cell A): Recreate engineered features on *_df_eng consistently (safe + no leakage)
# Run this AFTER you split (Section C) and AFTER you created training_df_eng/validation_df_eng/testing_df_eng copies.
# This ensures the engineered columns exist in ALL splits (not just in cleaned_df).

import numpy as np
import pandas as pd

if "training_df_eng" not in globals():
    print("Run Section C and the D copy cell first (training_df_eng not found).")
else:
    target_cols = [c for c in training_df_eng.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None

    def add_engineered_features(df):
        df = df.copy()

        # 1) log_total_spend
        if "total_spend" in df.columns:
            df["log_total_spend"] = np.log1p(df["total_spend"])
        else:
            df["log_total_spend"] = 0.0  # fallback (shouldn't happen)

        # 2) spend_per_order
        if ("total_spend" in df.columns) and ("n_orders" in df.columns):
            denom = df["n_orders"].replace(0, np.nan)
            df["spend_per_order"] = (df["total_spend"] / denom).fillna(0.0)
        else:
            df["spend_per_order"] = 0.0

        # 3) orders_per_year_active (time-normalised frequency proxy)
        # Uses recency_days as a simple time window proxy (already engineered earlier).
        if ("n_orders" in df.columns) and ("recency_days" in df.columns):
            years_active = (df["recency_days"] + 1) / 365.0
            df["orders_per_year_active"] = (df["n_orders"] / years_active).replace([np.inf, -np.inf], 0.0).fillna(0.0)
        else:
            df["orders_per_year_active"] = 0.0

        return df

    training_df_eng = add_engineered_features(training_df_eng)
    validation_df_eng = add_engineered_features(validation_df_eng)
    testing_df_eng = add_engineered_features(testing_df_eng)

    print("✅ Engineered features added to all splits.")
    print("Train columns now include:", [c for c in ["log_total_spend","spend_per_order","orders_per_year_active"] if c in training_df_eng.columns])
    print("Shapes:", training_df_eng.shape, validation_df_eng.shape, testing_df_eng.shape)

✅ Engineered features added to all splits.
Train columns now include: ['log_total_spend', 'spend_per_order', 'orders_per_year_active']
Shapes: (13383, 12) (2868, 12) (2868, 12)


In [92]:
# D.n Fix (Extra Cell B): Build X/y for each split from *_df_eng (drop ID + target)
# This is the leakage-safe way to create modelling inputs for later preprocessing.

import pandas as pd

if "training_df_eng" not in globals():
    print("Run the split + engineering cells first.")
else:
    target_cols = [c for c in training_df_eng.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None
    if target_col is None:
        raise ValueError("No y_* target column found in training_df_eng.")

    drop_cols = [c for c in ["customer_id", target_col] if c in training_df_eng.columns]

    X_train = training_df_eng.drop(columns=drop_cols).copy()
    y_train = training_df_eng[target_col].copy()

    X_val = validation_df_eng.drop(columns=[c for c in ["customer_id", target_col] if c in validation_df_eng.columns]).copy()
    y_val = validation_df_eng[target_col].copy()

    X_test = testing_df_eng.drop(columns=[c for c in ["customer_id", target_col] if c in testing_df_eng.columns]).copy()
    y_test = testing_df_eng[target_col].copy()

    print("✅ Created X/y splits (no leakage).")
    print("X_train:", X_train.shape, "y_train:", y_train.shape)
    print("X_val:  ", X_val.shape,   "y_val:  ", y_val.shape)
    print("X_test: ", X_test.shape,  "y_test: ", y_test.shape)

✅ Created X/y splits (no leakage).
X_train: (13383, 10) y_train: (13383,)
X_val:   (2868, 10) y_val:   (2868,)
X_test:  (2868, 10) y_test:  (2868,)


In [93]:
# D.n Fix (Extra Cell C): Enforce identical predictor columns across splits (and same order)
# This prevents downstream preprocessing/model errors.

if "X_train" not in globals():
    print("Run D.n Extra Cell B first to create X_train/X_val/X_test.")
else:
    train_cols = list(X_train.columns)
    val_cols = list(X_val.columns)
    test_cols = list(X_test.columns)

    missing_in_val = sorted(list(set(train_cols) - set(val_cols)))
    missing_in_test = sorted(list(set(train_cols) - set(test_cols)))
    extra_in_val = sorted(list(set(val_cols) - set(train_cols)))
    extra_in_test = sorted(list(set(test_cols) - set(train_cols)))

    print("Column consistency checks:")
    print("  Missing in val:", missing_in_val)
    print("  Missing in test:", missing_in_test)
    print("  Extra in val:", extra_in_val)
    print("  Extra in test:", extra_in_test)

    if missing_in_val or missing_in_test or extra_in_val or extra_in_test:
        raise ValueError("Feature columns are inconsistent across splits. Fix this before preprocessing.")

    # enforce same column order as training
    X_val = X_val[train_cols].copy()
    X_test = X_test[train_cols].copy()

    print("✅ Columns consistent and ordering enforced.")
    print("Final X shapes:", X_train.shape, X_val.shape, X_test.shape)

Column consistency checks:
  Missing in val: []
  Missing in test: []
  Extra in val: []
  Extra in test: []
✅ Columns consistent and ordering enforced.
Final X shapes: (13383, 10) (2868, 10) (2868, 10)


In [94]:
# D.n Fix (Extra Cell D): Quick leakage sanity check (IDs/target not in X, and target is binary)

if "X_train" not in globals():
    print("Run D.n Extra Cell B first.")
else:
    # Check ID/target not present
    bad_cols = [c for c in X_train.columns if (c == "customer_id") or c.startswith("y_")]
    print("Leakage-like columns in X_train (should be []):", bad_cols)

    # Check y is binary
    y_unique = set(pd.Series(y_train).dropna().unique())
    print("Unique y_train values:", y_unique)
    if not y_unique.issubset({0, 1}):
        raise ValueError("Target is not binary {0,1}. Ensure y_* was cleaned/cast correctly.")
    else:
        print("✅ Target is binary and X has no leakage columns.")

Leakage-like columns in X_train (should be []): []
Unique y_train values: {np.int64(0), np.int64(1)}
✅ Target is binary and X has no leakage columns.


In [95]:
feature_engineering_n_explanations = """
D.n Fix: Recompute engineered features consistently + enforce no-leakage X/y + align columns across splits

What the issue is
- Section D introduces engineered features and prepares data for modelling. At this stage, there are three common “pipeline-breaking” risks:
  1) Feature inconsistency across splits:
     - A new engineered column might be created in one split (e.g., training) but not in validation/testing, or vice versa.
     - This causes preprocessing/model code to fail or silently behave differently across splits.
  2) Leakage from identifiers or target into predictors:
     - customer_id is an identifier and must not be used as a predictor.
     - The target y_* must be separated from X before preprocessing/modelling.
  3) Column mismatch / ordering issues:
     - Even if the same columns exist, different column order across splits can break transformations or lead to incorrect feature mapping (especially once arrays/matrices are produced).

Why it is important
- Prevents invalid evaluation:
  - If customer_id or y_* leaks into X, validation/testing performance can be artificially high and not representative of real-world prediction.
- Ensures reproducible preprocessing:
  - Transformers (OneHotEncoder, scaling) expect the same input schema across train/val/test.
- Avoids runtime failures:
  - Many sklearn pipelines will error if validation/testing have missing columns compared to training.
- Guarantees that coefficients/feature importances map to the correct features:
  - Consistent column order is critical once the data is converted into numpy matrices.

How it was fixed
- Engineered-feature consistency:
  - Recomputed engineered features (e.g., log_total_spend, spend_per_order, orders_per_year_active) on training_df_eng, validation_df_eng, and testing_df_eng using the same formulas.
- Leakage guard:
  - Explicitly removed customer_id and the target y_* from predictors when constructing X_train/X_val/X_test.
  - Constructed y_train/y_val/y_test separately.
- Column alignment:
  - Verified that X_val and X_test contain exactly the same predictor columns as X_train.
  - Enforced identical column order based on the training set schema.

Impact
- Prevents accidental leakage and ensures the model is evaluated fairly.
- Makes downstream preprocessing (encoding/scaling) stable and compatible across all splits.
- Produces clean, consistent X/y inputs that can be safely used in the next sections for transformation and model training.
"""

In [96]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_n_explanations', value=feature_engineering_n_explanations)

---
## E. Data Preparation for Modeling

In [97]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  X_train = training_df_eng.copy()
  X_val = validation_df_eng.copy()
  X_test = testing_df_eng.copy()
except Exception as e:
  print(e)

### E.1 Data Transformation `One-Hot Encoding + Standard Scaling (train-fit only)`


In [98]:
# E.1 Data Transformation: One-Hot Encoding + Standard Scaling (fit on training only)
# Output: X_train_prepared, X_val_prepared, X_test_prepared, feature_names

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import pandas as pd
import numpy as np

# --- Preconditions ---
if "training_df_eng" not in globals():
    print("Run Section C (split) and Section D (engineering copies) first.")
else:
    # Detect target column (y_*)
    target_cols = [c for c in training_df_eng.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None
    if target_col is None:
        raise ValueError("No y_* target column found in training_df_eng.")

    # Build X/y from engineered splits (no leakage)
    drop_cols_train = [c for c in ["customer_id", target_col] if c in training_df_eng.columns]
    drop_cols_val   = [c for c in ["customer_id", target_col] if c in validation_df_eng.columns]
    drop_cols_test  = [c for c in ["customer_id", target_col] if c in testing_df_eng.columns]

    X_train = training_df_eng.drop(columns=drop_cols_train).copy()
    y_train = training_df_eng[target_col].copy()

    X_val = validation_df_eng.drop(columns=drop_cols_val).copy()
    y_val = validation_df_eng[target_col].copy()

    X_test = testing_df_eng.drop(columns=drop_cols_test).copy()
    y_test = testing_df_eng[target_col].copy()

    # Ensure consistent columns/order across splits
    train_cols = list(X_train.columns)
    if set(X_val.columns) != set(train_cols) or set(X_test.columns) != set(train_cols):
        missing_val = sorted(list(set(train_cols) - set(X_val.columns)))
        missing_test = sorted(list(set(train_cols) - set(X_test.columns)))
        extra_val = sorted(list(set(X_val.columns) - set(train_cols)))
        extra_test = sorted(list(set(X_test.columns) - set(train_cols)))
        raise ValueError(
            "Feature columns differ across splits.\n"
            f"Missing in val: {missing_val}\n"
            f"Missing in test: {missing_test}\n"
            f"Extra in val: {extra_val}\n"
            f"Extra in test: {extra_test}"
        )

    X_val = X_val[train_cols].copy()
    X_test = X_test[train_cols].copy()

    # Identify numeric vs categorical columns
    numeric_features = X_train.select_dtypes(include="number").columns.tolist()
    categorical_features = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()

    print("E.1 Column types:")
    print("  Numeric features:", len(numeric_features))
    print("  Categorical features:", len(categorical_features))

    # Build transformers
    numeric_transformer = Pipeline(steps=[
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        remainder="drop"
    )

    # Fit ONLY on training data (prevents leakage)
    preprocessor.fit(X_train)

    # Transform all splits
    X_train_prepared = preprocessor.transform(X_train)
    X_val_prepared = preprocessor.transform(X_val)
    X_test_prepared = preprocessor.transform(X_test)

    # Feature names for reporting
    feature_names = preprocessor.get_feature_names_out()

    print("\nPrepared shapes:")
    print("  X_train_prepared:", X_train_prepared.shape)
    print("  X_val_prepared:  ", X_val_prepared.shape)
    print("  X_test_prepared: ", X_test_prepared.shape)

    print("\nPrepared feature names:", len(feature_names))
    display(pd.Series(feature_names).head(30))

E.1 Column types:
  Numeric features: 9
  Categorical features: 1

Prepared shapes:
  X_train_prepared: (13383, 14)
  X_val_prepared:   (2868, 14)
  X_test_prepared:  (2868, 14)

Prepared feature names: 14


0                                     num__recency_days
1                                         num__n_orders
2                                      num__total_spend
3                                  num__avg_order_value
4                                  num__max_order_value
5                        num__territory_id__was_missing
6                                  num__log_total_spend
7                                  num__spend_per_order
8                           num__orders_per_year_active
9     cat__territory_id_0ad4c625-bb65-4376-8a41-0a65...
10    cat__territory_id_25d73ad2-9e13-410b-8b0d-5f26...
11    cat__territory_id_2ac923e9-3043-4f5e-bae0-ad28...
12    cat__territory_id_e0e3bac4-790e-4617-84b3-be0c...
13                            cat__territory_id_unknown
dtype: object

In [99]:
# E.1 Extra Cell 1: Sanity checks after preprocessing (no NaNs/inf, consistent dimensions)

import numpy as np

if "X_train_prepared" not in globals():
    print("Run E.1 preprocessing cell first to create X_*_prepared.")
else:
    def matrix_checks(name, X):
        X = np.asarray(X)
        print(f"\n{name}:")
        print("  shape:", X.shape)
        print("  dtype:", X.dtype)
        print("  NaNs:", int(np.isnan(X).sum()) if np.issubdtype(X.dtype, np.floating) else "n/a")
        print("  infs:", int(np.isinf(X).sum()) if np.issubdtype(X.dtype, np.floating) else "n/a")
        print("  min/max:", float(np.nanmin(X)), "/", float(np.nanmax(X)))

    matrix_checks("X_train_prepared", X_train_prepared)
    matrix_checks("X_val_prepared", X_val_prepared)
    matrix_checks("X_test_prepared", X_test_prepared)

    # Ensure same number of columns across splits
    assert X_train_prepared.shape[1] == X_val_prepared.shape[1] == X_test_prepared.shape[1], \
        "Prepared feature dimension mismatch across splits!"
    print("\n✅ Prepared matrices have consistent feature dimensions across splits.")


X_train_prepared:
  shape: (13383, 14)
  dtype: float64
  NaNs: 0
  infs: 0
  min/max: -1.829226250589279 / 23.07829231105344

X_val_prepared:
  shape: (2868, 14)
  dtype: float64
  NaNs: 0
  infs: 0
  min/max: -1.829226250589279 / 17.85684372249091

X_test_prepared:
  shape: (2868, 14)
  dtype: float64
  NaNs: 0
  infs: 0
  min/max: -1.829226250589279 / 23.07829231105344

✅ Prepared matrices have consistent feature dimensions across splits.


In [100]:
# E.1 Extra Cell 2: Verify scaling worked (numeric columns should be ~mean 0, std 1 in TRAIN only)

import numpy as np

if "preprocessor" not in globals():
    print("Run E.1 preprocessing cell first to create preprocessor.")
else:
    # indices: numeric features are transformed first in the ColumnTransformer
    n_num = len(numeric_features) if "numeric_features" in globals() else 0

    if n_num == 0:
        print("No numeric features detected; scaling check skipped.")
    else:
        Xtr = np.asarray(X_train_prepared)
        num_block = Xtr[:, :n_num]

        means = num_block.mean(axis=0)
        stds = num_block.std(axis=0)

        print("Numeric block (TRAIN) mean summary:")
        print("  mean(abs(mean)):", float(np.mean(np.abs(means))))
        print("  max(abs(mean)) :", float(np.max(np.abs(means))))

        print("Numeric block (TRAIN) std summary:")
        print("  mean(std):", float(np.mean(stds)))
        print("  min(std) :", float(np.min(stds)))
        print("  max(std) :", float(np.max(stds)))

        print("\n(Notes) Means should be close to 0 and stds close to 1 on TRAIN because scaler was fit on TRAIN.")

Numeric block (TRAIN) mean summary:
  mean(abs(mean)): 8.53466609714948e-17
  max(abs(mean)) : 4.903360588863706e-16
Numeric block (TRAIN) std summary:
  mean(std): 1.0000000000000078
  min(std) : 0.9999999999999915
  max(std) : 1.0000000000000777

(Notes) Means should be close to 0 and stds close to 1 on TRAIN because scaler was fit on TRAIN.


In [101]:
# E.1 Extra Cell 3: Inspect one-hot categories learned (helps reporting + debugging)

if "preprocessor" not in globals():
    print("Run E.1 preprocessing cell first.")
else:
    # Pull fitted one-hot encoder
    try:
        ohe = preprocessor.named_transformers_["cat"].named_steps["onehot"]
        cat_cols = categorical_features if "categorical_features" in globals() else []
    except Exception as e:
        print("Could not access OneHotEncoder:", e)
        ohe = None
        cat_cols = []

    if ohe is None or len(cat_cols) == 0:
        print("No categorical transformer present or no categorical features.")
    else:
        # categories_ is a list aligned with categorical_features
        cat_sizes = {col: len(cats) for col, cats in zip(cat_cols, ohe.categories_)}
        cat_sizes = pd.Series(cat_sizes).sort_values(ascending=False)

        print("One-hot category counts per column:")
        display(cat_sizes.to_frame("n_categories"))

        # show top categories for the first few categorical columns
        for col, cats in list(zip(cat_cols, ohe.categories_))[:3]:
            print(f"\nTop categories learned for {col} (first 15):")
            display(pd.Series(cats).head(15))

One-hot category counts per column:


,n_categories
territory_id,5



Top categories learned for territory_id (first 15):


0    0ad4c625-bb65-4376-8a41-0a65719b0db8
1    25d73ad2-9e13-410b-8b0d-5f26e2b9e4f2
2    2ac923e9-3043-4f5e-bae0-ad28089bf187
3    e0e3bac4-790e-4617-84b3-be0c2bdb7070
4                                 unknown
dtype: object

In [102]:
# E.1 Extra Cell 4: Convert prepared matrices to DataFrames with column names (optional, useful for inspection)

import pandas as pd
import numpy as np

if "feature_names" not in globals():
    print("Run E.1 preprocessing cell first to create feature_names.")
else:
    X_train_prepared_df = pd.DataFrame(X_train_prepared, columns=feature_names, index=X_train.index)
    X_val_prepared_df = pd.DataFrame(X_val_prepared, columns=feature_names, index=X_val.index)
    X_test_prepared_df = pd.DataFrame(X_test_prepared, columns=feature_names, index=X_test.index)

    print("✅ Created DataFrame versions with feature names.")
    display(X_train_prepared_df.head())
    print("X_train_prepared_df shape:", X_train_prepared_df.shape)

✅ Created DataFrame versions with feature names.


,num__recency_days,num__n_orders,num__total_spend,num__avg_order_value,num__max_order_value,num__territory_id__was_missing,num__log_total_spend,num__spend_per_order,num__orders_per_year_active,cat__territory_id_0ad4c625-bb65-4376-8a41-0a65719b0db8,cat__territory_id_25d73ad2-9e13-410b-8b0d-5f26e2b9e4f2,cat__territory_id_2ac923e9-3043-4f5e-bae0-ad28089bf187,cat__territory_id_e0e3bac4-790e-4617-84b3-be0c2bdb7070,cat__territory_id_unknown
0,0.681044,-0.548667,-0.206067,-0.247483,-0.245817,-1.037697,0.175549,-0.247020,-0.346340,0.0,0.0,0.0,1.0,0.0
1,0.100130,0.369184,0.170627,0.605390,0.421241,0.963672,1.210365,0.632242,-0.215670,0.0,0.0,0.0,0.0,1.0
2,-1.187678,-0.548667,-0.241226,-0.425518,-0.360430,0.963672,-0.764492,-0.430562,0.663850,0.0,0.0,0.0,0.0,1.0
3,-0.921718,0.369184,-0.027746,0.103148,0.171619,-1.037697,0.925125,0.114460,0.303158,0.0,0.0,0.0,1.0,0.0
4,0.562061,-0.548667,-0.241801,-0.428430,-0.362305,0.963672,-0.821606,-0.433565,-0.341797,0.0,0.0,0.0,0.0,1.0


X_train_prepared_df shape: (13383, 14)


In [103]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# expects: training_df_eng, validation_df_eng, testing_df_eng
target_cols = [c for c in training_df_eng.columns if c.startswith("y_")]
target_col = target_cols[0]

X_train = training_df_eng.drop(columns=[c for c in ["customer_id", target_col] if c in training_df_eng.columns]).copy()
y_train = training_df_eng[target_col].copy()

X_val = validation_df_eng.drop(columns=[c for c in ["customer_id", target_col] if c in validation_df_eng.columns]).copy()
y_val = validation_df_eng[target_col].copy()

X_test = testing_df_eng.drop(columns=[c for c in ["customer_id", target_col] if c in testing_df_eng.columns]).copy()
y_test = testing_df_eng[target_col].copy()

train_cols = list(X_train.columns)
X_val = X_val.reindex(columns=train_cols)
X_test = X_test.reindex(columns=train_cols)

numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("scaler", StandardScaler())]), numeric_features),
        ("cat", Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical_features),
    ],
    remainder="drop"
)

preprocessor.fit(X_train)
X_train_prepared = preprocessor.transform(X_train)
X_val_prepared = preprocessor.transform(X_val)
X_test_prepared = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

In [104]:
data_transformation_1_explanations = """
E.1 Data Transformation: One-Hot Encoding + Standard Scaling (fit on training only)

Why this transformation is needed
- Most machine learning models require numeric inputs. However, our dataset contains a mix of:
  - numeric predictors (e.g., recency_days, n_orders, total_spend),
  - categorical predictors (e.g., territory_id / unknown category flags).
- In addition, numeric variables are on very different scales (spend can be very large while n_orders is small). If left unscaled, some models can become overly influenced by large-magnitude features.

What was done
1) Prevent leakage by separating X and y
- The target column (y_*) was separated from the predictors before any transformation.
- customer_id was also excluded because it is an identifier, not a meaningful predictor.

2) Fit transformations on the training set only
- The preprocessing pipeline (encoding + scaling) was fit using only the training data.
- The fitted transformers were then applied to validation and testing.
- This avoids “peeking” at the validation/testing distributions during training-time preprocessing.

3) Handle numeric and categorical columns appropriately
- Numeric features were standardised using StandardScaler:
  - centres each feature to mean ~0 and scales to standard deviation ~1 (based on training data).
- Categorical features were one-hot encoded using OneHotEncoder(handle_unknown='ignore'):
  - converts each category into binary indicator columns,
  - prevents errors when unseen categories appear in validation/testing.

Impacts
- Produces fully numeric, model-ready matrices (X_train_prepared, X_val_prepared, X_test_prepared).
- Improves fairness and stability for models sensitive to feature scale (e.g., logistic regression, SVM, kNN).
- Makes evaluation more trustworthy by preventing preprocessing leakage from validation/testing data.
- Ensures consistent feature representation across all dataset splits, improving reproducibility.
"""

In [105]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_1_explanations', value=data_transformation_1_explanations)

### E.2 Data Transformation "`Log transform skewed monetary features`"

In [106]:
# E.2 Data Transformation: Log transform skewed monetary features (fit parameters on TRAIN only)
# - Applies log1p to spend/value columns to reduce right-skew.
# - Uses TRAIN-derived column list only (no peeking at val/test).
# Output: X_train_log, X_val_log, X_test_log

import numpy as np
import pandas as pd

if "X_train" not in globals():
    print("Run E.1 first to create X_train/X_val/X_test (raw predictors).")
else:
    # Decide which columns to log-transform (based on TRAIN only)
    # Rule: numeric + name contains spend/value/amount/price/total
    num_cols = X_train.select_dtypes(include="number").columns.tolist()
    log_cols = [c for c in num_cols if any(k in c.lower() for k in ["spend", "value", "amount", "price", "total", "due", "subtotal"])]

    # (Optional) also include any very-skewed numeric columns by a quick heuristic (TRAIN only)
    # If you want to keep it simple, comment this block out.
    for c in num_cols:
        if c in log_cols:
            continue
        s = X_train[c].dropna()
        if len(s) > 0:
            # crude skew proxy: (p99 / median) large => heavy right tail
            med = s.median()
            p99 = s.quantile(0.99)
            if med > 0 and (p99 / med) >= 25:
                log_cols.append(c)

    log_cols = sorted(list(dict.fromkeys(log_cols)))  # dedupe, keep stable order

    print("Numeric columns:", len(num_cols))
    print("Columns selected for log1p transform:", log_cols)

    def apply_log1p(df, cols):
        out = df.copy()
        for c in cols:
            # safety: ensure non-negative before log1p (should already be true after cleaning)
            out[c] = np.log1p(out[c].clip(lower=0))
        return out

    X_train_log = apply_log1p(X_train, log_cols)
    X_val_log = apply_log1p(X_val, log_cols)
    X_test_log = apply_log1p(X_test, log_cols)

    # Quick check
    if len(log_cols) > 0:
        c0 = log_cols[0]
        print(f"\nExample check for '{c0}':")
        display(pd.DataFrame({
            "train_before": X_train[c0].head(5).values,
            "train_after_log1p": X_train_log[c0].head(5).values
        }))
    else:
        print("No columns were transformed (no suitable monetary/skewed numeric columns detected).")

Numeric columns: 9
Columns selected for log1p transform: ['avg_order_value', 'log_total_spend', 'max_order_value', 'spend_per_order', 'total_spend']

Example check for 'avg_order_value':


,train_before,train_after_log1p
0,612.13690,6.418588
1,3200.66015,8.071425
2,71.79190,4.287605
3,1676.32275,7.424954
4,62.95190,4.158131


In [107]:
# E.2 Extra Cell 1: Compare distribution summary before vs after log transform (TRAIN only)

import numpy as np
import pandas as pd

if "X_train_log" not in globals():
    print("Run E.2 main cell first to create X_train_log/X_val_log/X_test_log.")
else:
    cols = log_cols if "log_cols" in globals() else []
    if len(cols) == 0:
        print("No log-transformed columns to audit.")
    else:
        rows = []
        for c in cols:
            b = X_train[c].dropna()
            a = X_train_log[c].dropna()
            rows.append({
                "feature": c,
                "before_min": float(b.min()),
                "before_median": float(b.median()),
                "before_p99": float(b.quantile(0.99)),
                "before_max": float(b.max()),
                "after_min": float(a.min()),
                "after_median": float(a.median()),
                "after_p99": float(a.quantile(0.99)),
                "after_max": float(a.max()),
            })
        audit_log = pd.DataFrame(rows).sort_values("before_p99", ascending=False)
        display(audit_log)

,feature,before_min,before_median,before_p99,before_max,after_min,after_median,after_p99,after_max
4,total_spend,5.514000,606.622900,132424.064034,132424.064034,1.873954,6.409554,11.793772,11.793772
2,max_order_value,5.514000,596.689000,40003.724088,40003.724088,1.873954,6.393071,10.596753,10.596753
0,avg_order_value,5.514000,596.689000,26111.291061,26111.291061,1.873954,6.393071,10.170161,10.170161
3,spend_per_order,5.514000,596.689000,16553.008004,33106.016008,1.873954,6.393071,9.714384,10.407501
1,log_total_spend,1.873954,6.409554,11.793772,11.793772,1.055689,2.002770,2.548959,2.548959


In [108]:
# E.2 Extra Cell 2: Simple skewness proxy before vs after (TRAIN only)
# (uses pandas skew; fine for reporting)

import pandas as pd

if "X_train_log" not in globals():
    print("Run E.2 main cell first.")
else:
    cols = log_cols if "log_cols" in globals() else []
    if len(cols) == 0:
        print("No log-transformed columns to compute skewness for.")
    else:
        sk = []
        for c in cols:
            sk.append({
                "feature": c,
                "skew_before": float(X_train[c].skew()),
                "skew_after": float(X_train_log[c].skew()),
            })
        skew_df = pd.DataFrame(sk).sort_values("skew_before", ascending=False)
        display(skew_df)

,feature,skew_before,skew_after
4,total_spend,7.636988,0.065983
3,spend_per_order,7.353083,-0.095783
2,max_order_value,6.952842,-0.055394
0,avg_order_value,6.564343,-0.087291
1,log_total_spend,0.065983,-0.455994


In [109]:
# E.2 Extra Cell 4: Rebuild the E.1 preprocessing pipeline using the log-transformed X's
# This is the recommended flow if you're using both transformations:
#   E.2 (log) -> E.1 (one-hot + scaling)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

if "X_train_log" not in globals():
    print("Run E.2 main cell first.")
else:
    Xtr = X_train_log.copy()
    Xva = X_val_log.copy()
    Xte = X_test_log.copy()

    numeric_features_2 = Xtr.select_dtypes(include="number").columns.tolist()
    categorical_features_2 = Xtr.select_dtypes(include=["object", "string", "category"]).columns.tolist()

    numeric_transformer_2 = Pipeline(steps=[
        ("scaler", StandardScaler())
    ])

    categorical_transformer_2 = Pipeline(steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    preprocessor_2 = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer_2, numeric_features_2),
            ("cat", categorical_transformer_2, categorical_features_2),
        ],
        remainder="drop"
    )

    preprocessor_2.fit(Xtr)  # fit TRAIN only

    X_train_prepared_2 = preprocessor_2.transform(Xtr)
    X_val_prepared_2 = preprocessor_2.transform(Xva)
    X_test_prepared_2 = preprocessor_2.transform(Xte)

    feature_names_2 = preprocessor_2.get_feature_names_out()

    print("Prepared shapes after E.2(log) + encoding/scaling:")
    print("  X_train_prepared_2:", X_train_prepared_2.shape)
    print("  X_val_prepared_2:  ", X_val_prepared_2.shape)
    print("  X_test_prepared_2: ", X_test_prepared_2.shape)
    print("Total features:", len(feature_names_2))
    display(pd.Series(feature_names_2).head(30))

Prepared shapes after E.2(log) + encoding/scaling:
  X_train_prepared_2: (13383, 14)
  X_val_prepared_2:   (2868, 14)
  X_test_prepared_2:  (2868, 14)
Total features: 14


0                                     num__recency_days
1                                         num__n_orders
2                                      num__total_spend
3                                  num__avg_order_value
4                                  num__max_order_value
5                        num__territory_id__was_missing
6                                  num__log_total_spend
7                                  num__spend_per_order
8                           num__orders_per_year_active
9     cat__territory_id_0ad4c625-bb65-4376-8a41-0a65...
10    cat__territory_id_25d73ad2-9e13-410b-8b0d-5f26...
11    cat__territory_id_2ac923e9-3043-4f5e-bae0-ad28...
12    cat__territory_id_e0e3bac4-790e-4617-84b3-be0c...
13                            cat__territory_id_unknown
dtype: object

In [110]:
import numpy as np

# expects: X_train, X_val, X_test from E.1
num_cols = X_train.select_dtypes(include="number").columns.tolist()
log_cols = [c for c in num_cols if any(k in c.lower() for k in ["spend", "value", "amount", "price", "total", "due", "subtotal"])]

for c in num_cols:
    if c in log_cols:
        continue
    s = X_train[c].dropna()
    if len(s) > 0:
        med = s.median()
        p99 = s.quantile(0.99)
        if med > 0 and (p99 / med) >= 25:
            log_cols.append(c)

log_cols = sorted(list(dict.fromkeys(log_cols)))

def apply_log1p(df, cols):
    out = df.copy()
    for c in cols:
        out[c] = np.log1p(out[c].clip(lower=0))
    return out

X_train_log = apply_log1p(X_train, log_cols)
X_val_log = apply_log1p(X_val, log_cols)
X_test_log = apply_log1p(X_test, log_cols)

In [111]:
data_transformation_2_explanations = """
E.2 Data Transformation: Log transform (log1p) for skewed monetary features

Why this transformation is needed
- Monetary/aggregate variables such as total_spend and order value features are typically highly right-skewed:
  a small number of customers have extremely large values compared to the majority.
- Skewed distributions can make modelling harder because:
  - extreme values can dominate the scale,
  - linear models can become sensitive to a few high-value customers,
  - scaling becomes less meaningful when the distribution is extremely long-tailed.

What was done
- Selected monetary / value-like numeric features based on the training set only.
- Applied log1p(x) = log(1 + x) to those columns:
  - compresses the long right tail,
  - keeps zero values valid (log1p(0)=0),
  - reduces the influence of extreme observations without dropping rows.

Leakage prevention
- The list of transformed columns was decided from the training data only.
- The same log transform was then applied consistently to validation and testing.

Impact
- Distributions become less skewed and more “model-friendly”.
- Reduces sensitivity to extreme monetary outliers.
- Often improves performance for linear/logistic models and makes scaling more stable.
"""

In [112]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_2_explanations', value=data_transformation_2_explanations)

### E.3 Data Transformation "`Select final modelling features + ensure consistent column order across splits`"


In [113]:
# E.3 Data Transformation: Select final modelling features + ensure consistent column order across splits
# Goal:
# - Keep only the columns you intend to model with (prevents accidental leakage/extra columns).
# - Ensure X_train/X_val/X_test have the exact same columns and order.

import pandas as pd

if "X_train" not in globals():
    print("Run D.1 (or your X/y split cell) first to create X_train/X_val/X_test.")
else:
    # --- Option A (recommended): use features_list if you already created it in A.z ---
    if "features_list" in globals():
        selected_features = [c for c in features_list if c in X_train.columns]
        print("Using features_list from A.z. Selected features:", selected_features)

    # --- Option B: fallback - use all columns currently in X_train ---
    else:
        selected_features = X_train.columns.tolist()
        print("features_list not found; using all current X_train columns.")

    # Defensive: remove any target-like columns if present
    suspicious = [c for c in selected_features if c.lower().startswith("y_") or "target" in c.lower() or "label" in c.lower()]
    if len(suspicious) > 0:
        print("Removing suspicious (potential leakage) columns:", suspicious)
        selected_features = [c for c in selected_features if c not in suspicious]

    # --- Apply selection + enforce identical columns across splits ---
    X_train_e3 = X_train[selected_features].copy()
    X_val_e3   = X_val.reindex(columns=selected_features).copy()
    X_test_e3  = X_test.reindex(columns=selected_features).copy()

    # Sanity checks
    print("\nE.3 Shapes after selection:")
    print("  X_train_e3:", X_train_e3.shape)
    print("  X_val_e3:  ", X_val_e3.shape)
    print("  X_test_e3: ", X_test_e3.shape)

    print("\nColumn equality checks:")
    print("  train==val columns:", list(X_train_e3.columns) == list(X_val_e3.columns))
    print("  train==test columns:", list(X_train_e3.columns) == list(X_test_e3.columns))

    # Optional: store back into the standard variable names for the next steps
    X_train = X_train_e3
    X_val = X_val_e3
    X_test = X_test_e3

    print("\nUpdated X_train/X_val/X_test to E.3-selected versions.")
    display(X_train.head())

Using features_list from A.z. Selected features: ['n_orders', 'total_spend', 'avg_order_value', 'max_order_value']

E.3 Shapes after selection:
  X_train_e3: (13383, 4)
  X_val_e3:   (2868, 4)
  X_test_e3:  (2868, 4)

Column equality checks:
  train==val columns: True
  train==test columns: True

Updated X_train/X_val/X_test to E.3-selected versions.


,n_orders,total_spend,avg_order_value,max_order_value
0,1,612.1369,612.13690,612.1369
1,2,6401.3203,3200.66015,3756.9890
2,1,71.7919,71.79190,71.7919
3,2,3352.6455,1676.32275,2580.1419
4,1,62.9519,62.95190,62.9519


In [114]:
# E.3 Extra Cell 1: Confirm there are no unexpected columns (IDs, target, duplicates)
# (helps for reporting + avoids silent leakage)

if "X_train" not in globals():
    print("Run E.3 first.")
else:
    bad_cols = [c for c in X_train.columns if c in ["customer_id"] or c.lower().startswith("y_")]
    print("Potentially problematic columns still in X_train:", bad_cols)

    # duplicates in columns
    dup_cols = pd.Index(X_train.columns).duplicated().sum()
    print("Duplicate column names in X_train:", int(dup_cols))

Potentially problematic columns still in X_train: []
Duplicate column names in X_train: 0


In [115]:
# expects: X_train, X_val, X_test
if "features_list" in globals():
    selected_features = [c for c in features_list if c in X_train.columns]
else:
    selected_features = X_train.columns.tolist()

suspicious = [c for c in selected_features if c.lower().startswith("y_") or "target" in c.lower() or "label" in c.lower()]
selected_features = [c for c in selected_features if c not in suspicious]

X_train_e3 = X_train[selected_features].copy()
X_val_e3 = X_val.reindex(columns=selected_features).copy()
X_test_e3 = X_test.reindex(columns=selected_features).copy()

X_train, X_val, X_test = X_train_e3, X_val_e3, X_test_e3

In [116]:
# E.3 Extra Cell 2: Explanation text for E.3 (paste into data_transformation_3_explanations)

data_transformation_3_explanations = """
E.3 Data Transformation: Final feature selection + column alignment across splits

Why this transformation is needed
- After cleaning, splitting, and feature engineering, it is easy to accidentally carry extra columns into modelling
  (e.g., identifiers, helper flags, or any leftover columns that were not intended as predictors).
- Many ML pipelines also require that training, validation, and testing have:
  - the exact same predictor columns,
  - in the exact same order.
  Otherwise, transformations or model prediction can fail or behave incorrectly.

What was done
- Selected the final set of predictor columns (preferably using the previously defined features_list).
- Explicitly removed any target-like columns (y_*) or obvious identifier columns to prevent leakage.
- Reindexed validation and testing to match the training columns exactly, guaranteeing consistent structure.

Impacts
- Prevents accidental leakage and makes the modelling step reproducible.
- Ensures consistent feature representation across splits (reduces runtime errors and evaluation mistakes).
- Creates clean X_train/X_val/X_test inputs ready for encoding/scaling and model training.
"""

In [117]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_3_explanations', value=data_transformation_3_explanations)

### E.n Fixing `Prevent train/val/test schema mismatch + final modelling-ready matrices (no leakage)`

> You can add more cells related to data preparation in this section

In [118]:
# E.n Fixing: Prevent train/val/test schema mismatch + final modelling-ready matrices (no leakage)
# Problem this fixes:
# - After cleaning/feature engineering, the set of columns can differ across splits (or ordering can differ),
#   causing preprocessing/model code to break or behave inconsistently.
# - We also want a single, leak-free pipeline that fits on TRAIN only and transforms VAL/TEST consistently.

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import pandas as pd
import numpy as np

if "training_df_eng" not in globals():
    print("Run Section D setup cell first (training_df_eng/validation_df_eng/testing_df_eng).")
else:
    # --- Identify target safely ---
    target_cols = [c for c in training_df_eng.columns if c.startswith("y_")]
    target_col = target_cols[0] if len(target_cols) else None
    if target_col is None:
        raise ValueError("No y_* target column found. Create the label in Section A first.")

    # --- Separate X/y and drop identifiers ---
    drop_id_cols = [c for c in ["customer_id"] if c in training_df_eng.columns]

    X_train_raw = training_df_eng.drop(columns=drop_id_cols + [target_col], errors="ignore").copy()
    y_train = training_df_eng[target_col].copy()

    X_val_raw = validation_df_eng.drop(columns=drop_id_cols + [target_col], errors="ignore").copy()
    y_val = validation_df_eng[target_col].copy()

    X_test_raw = testing_df_eng.drop(columns=drop_id_cols + [target_col], errors="ignore").copy()
    y_test = testing_df_eng[target_col].copy()

    # --- Fix schema mismatch: force VAL/TEST to match TRAIN columns exactly ---
    train_cols = X_train_raw.columns.tolist()
    X_val_raw = X_val_raw.reindex(columns=train_cols)
    X_test_raw = X_test_raw.reindex(columns=train_cols)

    # --- Split columns by dtype (based on TRAIN only) ---
    numeric_features = X_train_raw.select_dtypes(include="number").columns.tolist()
    categorical_features = X_train_raw.select_dtypes(include=["object", "string", "category"]).columns.tolist()

    # --- Build leak-free preprocessing pipeline (fit on TRAIN only) ---
    numeric_transformer = Pipeline(steps=[
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        remainder="drop"
    )

    preprocessor.fit(X_train_raw)

    # --- Transform all splits using the fitted TRAIN pipeline ---
    X_train_prepared = preprocessor.transform(X_train_raw)
    X_val_prepared = preprocessor.transform(X_val_raw)
    X_test_prepared = preprocessor.transform(X_test_raw)

    # --- Feature names (for reporting) ---
    feature_names = preprocessor.get_feature_names_out()

    # --- Sanity checks ---
    print("E.n Fix complete: schema alignment + preprocessing (fit on train only).")
    print("Raw shapes:")
    print("  X_train_raw:", X_train_raw.shape, "y_train:", y_train.shape)
    print("  X_val_raw:  ", X_val_raw.shape,   "y_val:  ", y_val.shape)
    print("  X_test_raw: ", X_test_raw.shape,  "y_test: ", y_test.shape)

    print("\nPrepared shapes:")
    print("  X_train_prepared:", X_train_prepared.shape)
    print("  X_val_prepared:  ", X_val_prepared.shape)
    print("  X_test_prepared: ", X_test_prepared.shape)

    print("\nPrepared feature count:", len(feature_names))
    display(pd.Series(feature_names).head(30))

    # Optional: store as DataFrames (easier to inspect/export)
    X_train_prepared_df = pd.DataFrame(X_train_prepared, columns=feature_names)
    X_val_prepared_df = pd.DataFrame(X_val_prepared, columns=feature_names)
    X_test_prepared_df = pd.DataFrame(X_test_prepared, columns=feature_names)

    display(X_train_prepared_df.head())

E.n Fix complete: schema alignment + preprocessing (fit on train only).
Raw shapes:
  X_train_raw: (13383, 10) y_train: (13383,)
  X_val_raw:   (2868, 10) y_val:   (2868,)
  X_test_raw:  (2868, 10) y_test:  (2868,)

Prepared shapes:
  X_train_prepared: (13383, 14)
  X_val_prepared:   (2868, 14)
  X_test_prepared:  (2868, 14)

Prepared feature count: 14


0                                     num__recency_days
1                                         num__n_orders
2                                      num__total_spend
3                                  num__avg_order_value
4                                  num__max_order_value
5                        num__territory_id__was_missing
6                                  num__log_total_spend
7                                  num__spend_per_order
8                           num__orders_per_year_active
9     cat__territory_id_0ad4c625-bb65-4376-8a41-0a65...
10    cat__territory_id_25d73ad2-9e13-410b-8b0d-5f26...
11    cat__territory_id_2ac923e9-3043-4f5e-bae0-ad28...
12    cat__territory_id_e0e3bac4-790e-4617-84b3-be0c...
13                            cat__territory_id_unknown
dtype: object

,num__recency_days,num__n_orders,num__total_spend,num__avg_order_value,num__max_order_value,num__territory_id__was_missing,num__log_total_spend,num__spend_per_order,num__orders_per_year_active,cat__territory_id_0ad4c625-bb65-4376-8a41-0a65719b0db8,cat__territory_id_25d73ad2-9e13-410b-8b0d-5f26e2b9e4f2,cat__territory_id_2ac923e9-3043-4f5e-bae0-ad28089bf187,cat__territory_id_e0e3bac4-790e-4617-84b3-be0c2bdb7070,cat__territory_id_unknown
0,0.681044,-0.548667,-0.206067,-0.247483,-0.245817,-1.037697,0.175549,-0.247020,-0.346340,0.0,0.0,0.0,1.0,0.0
1,0.100130,0.369184,0.170627,0.605390,0.421241,0.963672,1.210365,0.632242,-0.215670,0.0,0.0,0.0,0.0,1.0
2,-1.187678,-0.548667,-0.241226,-0.425518,-0.360430,0.963672,-0.764492,-0.430562,0.663850,0.0,0.0,0.0,0.0,1.0
3,-0.921718,0.369184,-0.027746,0.103148,0.171619,-1.037697,0.925125,0.114460,0.303158,0.0,0.0,0.0,1.0,0.0
4,0.562061,-0.548667,-0.241801,-0.428430,-0.362305,0.963672,-0.821606,-0.433565,-0.341797,0.0,0.0,0.0,0.0,1.0


In [119]:
# E.n Extra Cell 1: Schema + leakage sanity audit (IDs/target not in X, matching columns across splits)

import pandas as pd

if "X_train_raw" not in globals():
    print("Run the E.n main fixing cell first to create X_train_raw/X_val_raw/X_test_raw.")
else:
    def schema_report(name, X, y=None):
        rep = {
            "split": name,
            "n_rows": int(X.shape[0]),
            "n_cols": int(X.shape[1]),
            "n_numeric_cols": int(len(X.select_dtypes(include="number").columns)),
            "n_categorical_cols": int(len(X.select_dtypes(include=["object","string","category"]).columns)),
            "has_customer_id_col": ("customer_id" in X.columns),
            "has_y_col_in_X": any([c.startswith("y_") for c in X.columns]),
        }
        if y is not None:
            rep["y_unique"] = sorted(pd.Series(y).dropna().unique().tolist())
            rep["y_mean"] = float(pd.Series(y).mean())
        return rep

    report = pd.DataFrame([
        schema_report("train", X_train_raw, y_train),
        schema_report("val", X_val_raw, y_val),
        schema_report("test", X_test_raw, y_test),
    ])
    display(report)

    print("Column match checks:")
    print("  train==val:", list(X_train_raw.columns) == list(X_val_raw.columns))
    print("  train==test:", list(X_train_raw.columns) == list(X_test_raw.columns))

,split,n_rows,n_cols,n_numeric_cols,n_categorical_cols,has_customer_id_col,has_y_col_in_X,y_unique,y_mean
0,train,13383,10,9,1,False,False,"[0, 1]",0.097362
1,val,2868,10,9,1,False,False,"[0, 1]",0.094142
2,test,2868,10,9,1,False,False,"[0, 1]",0.106695


Column match checks:
  train==val: True
  train==test: True


In [120]:
# E.n Extra Cell 2: Check for missing values created by reindexing (VAL/TEST) + fill safely (TRAIN stats only)
# This fixes the case where VAL/TEST had a column missing that TRAIN had; reindex will create NaNs.

import numpy as np
import pandas as pd

if "X_train_raw" not in globals():
    print("Run the E.n main fixing cell first.")
else:
    # Identify any columns where VAL/TEST now have NaNs due to reindexing or earlier steps
    val_nan_cols = [c for c in X_val_raw.columns if X_val_raw[c].isna().any()]
    test_nan_cols = [c for c in X_test_raw.columns if X_test_raw[c].isna().any()]
    nan_cols = sorted(set(val_nan_cols + test_nan_cols))

    print("Columns with NaNs in VAL or TEST:", nan_cols)

    if len(nan_cols) == 0:
        print("No NaNs introduced by alignment (good).")
    else:
        # Build TRAIN-only fill values
        fill_map = {}
        for c in nan_cols:
            if pd.api.types.is_numeric_dtype(X_train_raw[c]):
                fill_map[c] = float(X_train_raw[c].median())
            else:
                fill_map[c] = "unknown"

        # Apply fills
        X_val_raw = X_val_raw.copy()
        X_test_raw = X_test_raw.copy()
        for c, v in fill_map.items():
            X_val_raw[c] = X_val_raw[c].fillna(v)
            X_test_raw[c] = X_test_raw[c].fillna(v)

        print("Filled NaNs using TRAIN-only stats/tokens.")
        print("Remaining null cells:")
        print("  val:", int(X_val_raw.isna().sum().sum()))
        print("  test:", int(X_test_raw.isna().sum().sum()))

Columns with NaNs in VAL or TEST: []
No NaNs introduced by alignment (good).


In [121]:
# E.n Extra Cell 3: Re-transform after the NaN-fix (only if you ran Extra Cell 2)
# (keeps leakage-free: preprocessor is already fit on TRAIN)

import pandas as pd

if "preprocessor" not in globals():
    print("Run the E.n main fixing cell first to create 'preprocessor'.")
elif "X_val_raw" not in globals():
    print("Run E.n main cell first.")
else:
    X_train_prepared = preprocessor.transform(X_train_raw)
    X_val_prepared = preprocessor.transform(X_val_raw)
    X_test_prepared = preprocessor.transform(X_test_raw)

    feature_names = preprocessor.get_feature_names_out()

    print("Prepared shapes (after alignment + optional NaN fix):")
    print("  train:", X_train_prepared.shape)
    print("  val:  ", X_val_prepared.shape)
    print("  test: ", X_test_prepared.shape)

    # Optional DataFrame versions
    X_train_prepared_df = pd.DataFrame(X_train_prepared, columns=feature_names)
    X_val_prepared_df = pd.DataFrame(X_val_prepared, columns=feature_names)
    X_test_prepared_df = pd.DataFrame(X_test_prepared, columns=feature_names)

    display(X_train_prepared_df.head())

Prepared shapes (after alignment + optional NaN fix):
  train: (13383, 14)
  val:   (2868, 14)
  test:  (2868, 14)


,num__recency_days,num__n_orders,num__total_spend,num__avg_order_value,num__max_order_value,num__territory_id__was_missing,num__log_total_spend,num__spend_per_order,num__orders_per_year_active,cat__territory_id_0ad4c625-bb65-4376-8a41-0a65719b0db8,cat__territory_id_25d73ad2-9e13-410b-8b0d-5f26e2b9e4f2,cat__territory_id_2ac923e9-3043-4f5e-bae0-ad28089bf187,cat__territory_id_e0e3bac4-790e-4617-84b3-be0c2bdb7070,cat__territory_id_unknown
0,0.681044,-0.548667,-0.206067,-0.247483,-0.245817,-1.037697,0.175549,-0.247020,-0.346340,0.0,0.0,0.0,1.0,0.0
1,0.100130,0.369184,0.170627,0.605390,0.421241,0.963672,1.210365,0.632242,-0.215670,0.0,0.0,0.0,0.0,1.0
2,-1.187678,-0.548667,-0.241226,-0.425518,-0.360430,0.963672,-0.764492,-0.430562,0.663850,0.0,0.0,0.0,0.0,1.0
3,-0.921718,0.369184,-0.027746,0.103148,0.171619,-1.037697,0.925125,0.114460,0.303158,0.0,0.0,0.0,1.0,0.0
4,0.562061,-0.548667,-0.241801,-0.428430,-0.362305,0.963672,-0.821606,-0.433565,-0.341797,0.0,0.0,0.0,0.0,1.0


In [122]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import pandas as pd

# expects: training_df_eng, validation_df_eng, testing_df_eng
target_col = [c for c in training_df_eng.columns if c.startswith("y_")][0]

X_train_raw = training_df_eng.drop(columns=[c for c in ["customer_id", target_col] if c in training_df_eng.columns], errors="ignore").copy()
y_train = training_df_eng[target_col].copy()

X_val_raw = validation_df_eng.drop(columns=[c for c in ["customer_id", target_col] if c in validation_df_eng.columns], errors="ignore").copy()
y_val = validation_df_eng[target_col].copy()

X_test_raw = testing_df_eng.drop(columns=[c for c in ["customer_id", target_col] if c in testing_df_eng.columns], errors="ignore").copy()
y_test = testing_df_eng[target_col].copy()

train_cols = X_train_raw.columns.tolist()
X_val_raw = X_val_raw.reindex(columns=train_cols)
X_test_raw = X_test_raw.reindex(columns=train_cols)

numeric_features = X_train_raw.select_dtypes(include="number").columns.tolist()
categorical_features = X_train_raw.select_dtypes(include=["object", "string", "category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("scaler", StandardScaler())]), numeric_features),
        ("cat", Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical_features),
    ],
    remainder="drop"
)

preprocessor.fit(X_train_raw)
X_train_prepared = preprocessor.transform(X_train_raw)
X_val_prepared = preprocessor.transform(X_val_raw)
X_test_prepared = preprocessor.transform(X_test_raw)
feature_names = preprocessor.get_feature_names_out()

X_train_prepared_df = pd.DataFrame(X_train_prepared, columns=feature_names)
X_val_prepared_df = pd.DataFrame(X_val_prepared, columns=feature_names)
X_test_prepared_df = pd.DataFrame(X_test_prepared, columns=feature_names)

In [123]:
data_transformation_n_explanations = """
E.n Data Transformation / Fix: Ensure consistent, leak-free preprocessing across train/validation/test

Why this transformation is important
- After splitting the dataset, the training/validation/testing sets must have the same predictor columns in the same order.
  If the schemas differ (missing columns, extra columns, or different ordering), preprocessing and model training can fail
  or produce incorrect results.
- It is also critical to avoid data leakage: any preprocessing parameters (e.g., scaling means/standard deviations or
  one-hot encoding category mappings) must be learned from the training set only. Using validation/testing to fit these
  transformations would leak information from the evaluation sets into training and inflate performance.

What was done (method)
1) Predictor/target separation (leakage prevention)
   - The target column (y_*) was separated into y_train / y_val / y_test.
   - Identifier columns (e.g., customer_id) were removed from X to prevent the model memorising IDs.

2) Schema alignment across splits
   - The training predictors (X_train) were treated as the “source of truth” schema.
   - Validation and testing predictors were reindexed to match X_train’s columns exactly.
   - This guarantees consistent input structure for the pipeline and avoids silent column mismatches.

3) Training-only fit preprocessing pipeline
   - Numeric features were standardised using StandardScaler (fit on training only).
   - Categorical features were one-hot encoded using OneHotEncoder(handle_unknown='ignore') so unseen categories in
     validation/testing do not break the transformation.
   - The fitted preprocessor was then applied to validation and testing without refitting.

Impacts / outputs
- Produces modelling-ready matrices:
  - X_train_prepared, X_val_prepared, X_test_prepared (all numeric and consistent across splits)
  - y_train, y_val, y_test (targets kept separate)
- Improves reproducibility and correctness of evaluation by ensuring that validation/testing performance reflects true
  generalisation rather than preprocessing leakage or schema inconsistencies.
"""

In [124]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_n_explanations', value=data_transformation_n_explanations)

In [125]:
from IPython.display import display, HTML
import html

html_text = "<pre style='white-space: pre-wrap; word-wrap: break-word; font-size: 13px; line-height: 1.35;'>" \
            + html.escape(data_transformation_n_explanations.strip()) + \
            "</pre>"

display(HTML(html_text))

---
## F. Save Datasets

> Do not change this code

In [126]:
# DO NOT MODIFY THE CODE IN THIS CELL

try:
  X_train.to_csv(at.folder_path / 'X_train.csv', index=False)

  X_val.to_csv(at.folder_path / 'X_val.csv', index=False)

  X_test.to_csv(at.folder_path / 'X_test.csv', index=False)
except Exception as e:
  print(e)